

Pulls from `/api/monitor/*` --> DataFrame

**Derived tables** 
(`dashboard/models.py`) — `MetricsDaily`, `MetricsParticipant`,   `MetricsCohort`, `Alert`. 

 They hold no collected data and are empty until `python manage.py recompute_metrics --all` has run against this database.
 
  **Raw source tables** (`backend/app/models.py`) -- the ten `app_*` tables the   `dashboard/data/*` modules actually read.

`evaluate_alerts`, `refresh_risk_scores` and `recompute_metrics`write to the database and are never called here — `backend/.env` can point at production.

In [267]:
# Django bootstrap

import json
import os
import sys
from datetime import timedelta
from pathlib import Path

import pandas as pd

os.environ.setdefault("DJANGO_ALLOW_ASYNC_UNSAFE", "true")

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "dashboard" else CWD
BACKEND_DIR = REPO_ROOT / "backend"
for _path in (REPO_ROOT, BACKEND_DIR):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "project.settings")

import django

django.setup()

from django.conf import settings
from django.utils import timezone

_db = settings.DATABASES["default"]
print("repo root:", REPO_ROOT)
print("engine:   ", _db["ENGINE"])
print("host:     ", _db.get("HOST") or "(local)")
print("database: ", _db.get("NAME"))

repo root: /Users/tienle/Documents/Coding/REACT-SMASH/REACT
engine:    django.db.backends.postgresql
host:      c5dqhursbgn9fb.cluster-czrs8kj4isg7.us-east-1.rds.amazonaws.com
database:  dd28mps21fsprq


In [268]:
SYNTHETIC_DATA = True

# When True every frame comes from the committed fixture and nothing touches the
# database - the notebook runs with no connection at all. When False the ORM path
# below is used unchanged. Regenerate with: python dashboard/make_fixture.py
from dashboard.data.config import PARTICIPANT_TZ

FIXTURE_PATH = REPO_ROOT / "dashboard" / "fixture_cohort.json"
HR_DAYS = 14

_FIXTURE = None

UTC_COLUMNS = {
    "app_user": ("enrolled_at",),
    "app_wearabledevice": ("last_synced_at",),
    "app_wearablesync": ("observed_at", "last_synced_at"),
    "app_heartratesample": ("timestamp",),
    "app_ema": ("sent_at", "responded_at", "expires_at",
                "outcome_window_start", "outcome_window_end"),
    "app_checkinreminder": ("sent_at",),
    "app_phonetelemetry": ("occurred_at", "recorded_at"),
    "app_jitailog": ("triggered_at", "decision_made_at", "push_sent_at",
                     "device_received_at", "receipt_reported_at"),
    "app_engagementlog": ("occurred_at", "recorded_at"),
    "dashboard_metricsdaily": ("computed_at",),
    "dashboard_metricsparticipant": ("computed_at", "enrolled_at",
                                     "first_seen_not_enrolled_at", "last_ema_at", "last_sync_at"),
    "dashboard_metricscohort": ("as_of",),
    "dashboard_alert": ("fired_at", "resolved_at"),
}

# Compared against today_local() results, which are datetime.date. A Timestamp
# here would silently break every participant-day join.
DATE_COLUMNS = {
    "app_user": ("birthdate",),
    "dashboard_metricsdaily": ("local_date",),
    "dashboard_metricsparticipant": ("day1_date",),
}

BOOL_COLUMNS = {
    "app_user": ("is_enrolled",),
    "app_wearabledevice": ("is_active",),
    "app_jitailog": ("send_prompt",),
    "dashboard_metricsdaily": ("is_run_in", "is_active_day", "cap_hit"),
    "dashboard_metricsparticipant": ("is_enrolled_snapshot", "active_retention"),
}


def load_fixture():
    global _FIXTURE
    if _FIXTURE is None:
        with open(FIXTURE_PATH, encoding="utf-8") as handle:
            _FIXTURE = json.load(handle)
        _FIXTURE["tables"]["app_heartratesample"] = expand_heart_rate(_FIXTURE)
    return _FIXTURE


def expand_heart_rate(fixture):
    """Run-length encoded worn stretches -> one row per minute.

    Stored compressed because 1-minute fidelity is what wear_lane bins on and
    what wear_coverage compares gap-coverage against; a coarser cadence would
    flag every day as burst charging.
    """
    rows = []
    sample_id = 0
    for day in fixture.get("heart_rate_runs", []):
        local_date = pd.Timestamp(day["local_date"])
        midnight = local_date.tz_localize(PARTICIPANT_TZ)
        for start_minute, series in day["runs"]:
            for offset, bpm in enumerate(series):
                sample_id += 1
                rows.append({
                    "id": sample_id,
                    "user_id": day["user_id"],
                    "timestamp": (midnight + pd.Timedelta(minutes=start_minute + offset)
                                  ).tz_convert("UTC").isoformat(),
                    "bpm": bpm,
                    "source": "garmin_labfront",
                })
    return rows


def coerce(table, df):
    for column in UTC_COLUMNS.get(table, ()):
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], utc=True, errors="coerce")
    for column in DATE_COLUMNS.get(table, ()):
        if column in df.columns:
            df[column] = [None if pd.isna(v) else pd.Timestamp(v).date() for v in df[column]]
    for column in BOOL_COLUMNS.get(table, ()):
        if column in df.columns:
            df[column] = df[column].fillna(False).astype(bool)
    return df


def fixture_table(table, fields):
    rows = load_fixture()["tables"].get(table, [])
    return coerce(table, pd.DataFrame(rows, columns=list(fields)))


if SYNTHETIC_DATA:
    meta = load_fixture()["meta"]
    print("SOURCE: fixture", FIXTURE_PATH.name)
    for key in ("generated_from", "seed", "n_users", "anchor_date", "item_bank_version"):
        if key in meta:
            print(f"  {key:20s} {meta[key]}")
    print(f"  {'note':20s} {meta['note']}")
else:
    print("SOURCE: Django ORM ->", _db.get("HOST") or "(local)", "/", _db.get("NAME"))

SOURCE: fixture fixture_cohort.json
  seed                 17
  n_users              14
  anchor_date          2026-09-16
  item_bank_version    v1
  note                 Synthetic. The four dashboard_* tables are fabricated, not produced by recompute_metrics; only MetricsParticipant.first_seen_not_enrolled_at and Alert are read by the notebook's own computations.


In [269]:
# LOADER HELPERS
def frame(queryset, *fields):
    """One frame per table, from the fixture or the ORM.

    Dispatches on the model's own db_table so the ten loader cells below are
    identical in both modes. Django querysets are lazy, so constructing one with
    no database behind it is safe - nothing is evaluated until .values() runs,
    which the synthetic branch never reaches.
    """
    table = queryset.model._meta.db_table
    if SYNTHETIC_DATA:
        return fixture_table(table, fields)
    return pd.DataFrame(list(queryset.values(*fields)), columns=list(fields))


def shape(name, df):
    return {"table": name, "rows": len(df), "columns": df.shape[1]}

In [270]:
# STUDY CONSTANTS
from dashboard.data.config import (
    BENCHMARKS,
    DAILY_PROMPT_CAP,
    ITEM_BANK_VERSION,
    JITAI_COOLDOWN_MINUTES,
    MSSD_WINDOW,
    OUTCOME_WINDOW_HOURS,
    PARTICIPANT_TZ,
    RATE_MIN_PARTICIPANTS,
    RATE_MIN_UNITS,
    RUN_IN_DAYS,
    STUDY_DAYS,
    THRESHOLD_QUANTILE,
    WAKING_WINDOW_END_HOUR,
    WAKING_WINDOW_START_HOUR,
)

constants = {
    "STUDY_DAYS": STUDY_DAYS,
    "RUN_IN_DAYS": RUN_IN_DAYS,
    "PARTICIPANT_TZ": str(PARTICIPANT_TZ),
    "WAKING_WINDOW_START_HOUR": WAKING_WINDOW_START_HOUR,
    "WAKING_WINDOW_END_HOUR": WAKING_WINDOW_END_HOUR,
    "JITAI_COOLDOWN_MINUTES": JITAI_COOLDOWN_MINUTES,
    "DAILY_PROMPT_CAP": DAILY_PROMPT_CAP,
    "THRESHOLD_QUANTILE": THRESHOLD_QUANTILE,
    "MSSD_WINDOW": MSSD_WINDOW,
    "OUTCOME_WINDOW_HOURS": OUTCOME_WINDOW_HOURS,
    "RATE_MIN_PARTICIPANTS": RATE_MIN_PARTICIPANTS,
    "RATE_MIN_UNITS": RATE_MIN_UNITS,
    "ITEM_BANK_VERSION": ITEM_BANK_VERSION,
    "BENCHMARKS": BENCHMARKS,
}
for key, value in constants.items():
    print(f"{key:26s} {value}")

STUDY_DAYS                 35
RUN_IN_DAYS                7
PARTICIPANT_TZ             America/New_York
WAKING_WINDOW_START_HOUR   8
WAKING_WINDOW_END_HOUR     22
JITAI_COOLDOWN_MINUTES     60
DAILY_PROMPT_CAP           4
THRESHOLD_QUANTILE         0.8
MSSD_WINDOW                3
OUTCOME_WINDOW_HOURS       2
RATE_MIN_PARTICIPANTS      10
RATE_MIN_UNITS             30
ITEM_BANK_VERSION          v1
BENCHMARKS                 {'slot_coverage': 0.75, 'prompt_response': 0.7, 'wear': 0.8, 'retention': 0.85}


##  Derived metric tables (`dashboard_*`)

In [271]:
from dashboard.models import Alert, MetricsCohort, MetricsDaily, MetricsParticipant

METRICS_DAILY_FIELDS = (
    "id", "user_id", "study_day", "local_date", "is_run_in", "is_active_day",
    "computed_at", "item_bank_version",
    "ema_scheduled_n", "ema_jitai_n", "ema_post_prompt_n",
    "slots_expected", "slots_covered", "slots_reminded_uncovered", "slots_silent",
    "reminders_sent", "reminders_per_checkin_median",
    "completeness_mean", "ema_missing_b1b2_n",
    "decision_points_n", "eligible_n", "sent_n", "delivered_n", "cap_hit",
    "min_gap_min", "cooldown_violations_n", "runin_violation_n",
    "prompt_opened_n", "prompt_acted_n", "prompt_dismissed_n", "outcome_captured_n",
    "wear_valid_pct", "wear_gap_pct", "gaps_gt2h_n", "max_gap_min", "hr_minutes_valid",
    "last_sync_age_h_eod", "clock_skew_p95_ms", "delivery_failures_n",
)

metrics_daily_df = frame(
    MetricsDaily.objects.order_by("user_id", "study_day"),
    *METRICS_DAILY_FIELDS,
)
metrics_daily_df.head()

,id,user_id,study_day,local_date,is_run_in,is_active_day,computed_at,item_bank_version,ema_scheduled_n,ema_jitai_n,...,prompt_dismissed_n,outcome_captured_n,wear_valid_pct,wear_gap_pct,gaps_gt2h_n,max_gap_min,hr_minutes_valid,last_sync_age_h_eod,clock_skew_p95_ms,delivery_failures_n
0,1,1001,0,2026-08-13,True,True,2026-09-16 10:00:00+00:00,v1,3,0,...,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,0
1,2,1001,1,2026-08-14,True,True,2026-09-16 10:00:00+00:00,v1,2,0,...,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,0
2,3,1001,2,2026-08-15,True,True,2026-09-16 10:00:00+00:00,v1,2,0,...,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,0
3,4,1001,3,2026-08-16,True,True,2026-09-16 10:00:00+00:00,v1,2,0,...,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,0
4,5,1001,4,2026-08-17,True,True,2026-09-16 10:00:00+00:00,v1,4,0,...,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,0


One row per participant per study day. **A null here is structural, not zero**: a day outside theparticipant's active range carries `is_active_day=False` with every metric `NULL`, while a dayinside it with no activity carries `0`. Every metric column is nullable for exactly that reason —do not `fillna(0)`.

In [272]:
METRICS_PARTICIPANT_FIELDS = (
    "id", "user_id", "computed_at", "enrolled_at", "day1_date", "study_day_now", "phase",
    "is_enrolled_snapshot", "first_seen_not_enrolled_at", "last_ema_at", "last_sync_at",
    "active_retention", "risk_score", "risk_components",
    "slot_coverage_rate", "slot_coverage_num", "slot_coverage_den",
    "prompt_response_rate", "prompt_response_num", "prompt_response_den",
    "wear_rate", "wear_num", "wear_den",
)

metrics_participant_df = frame(
    MetricsParticipant.objects.order_by("user_id"),
    *METRICS_PARTICIPANT_FIELDS,
)
metrics_participant_df.head()

,id,user_id,computed_at,enrolled_at,day1_date,study_day_now,phase,is_enrolled_snapshot,first_seen_not_enrolled_at,last_ema_at,...,risk_components,slot_coverage_rate,slot_coverage_num,slot_coverage_den,prompt_response_rate,prompt_response_num,prompt_response_den,wear_rate,wear_num,wear_den
0,1,1001,2026-09-16 10:00:00+00:00,2026-08-13 14:00:00+00:00,2026-08-13,34,complete,True,NaT,2026-09-15 21:00:00+00:00,...,NaN,NaN,0,0,NaN,0,0,NaN,0,0
1,2,1002,2026-09-16 10:00:00+00:00,2026-08-26 14:00:00+00:00,2026-08-26,21,withdrawn,False,2026-09-10 10:00:00+00:00,2026-09-15 21:00:00+00:00,...,NaN,NaN,0,0,NaN,0,0,NaN,0,0
2,3,1003,2026-09-16 10:00:00+00:00,2026-08-29 14:00:00+00:00,2026-08-29,18,mrt,True,NaT,2026-09-15 21:00:00+00:00,...,NaN,NaN,0,0,NaN,0,0,NaN,0,0
3,4,1004,2026-09-16 10:00:00+00:00,2026-09-10 14:00:00+00:00,2026-09-10,6,run_in,True,NaT,2026-09-15 21:00:00+00:00,...,NaN,NaN,0,0,NaN,0,0,NaN,0,0
4,5,1005,2026-09-16 10:00:00+00:00,2026-09-10 14:00:00+00:00,2026-09-10,6,run_in,True,NaT,2026-09-15 21:00:00+00:00,...,NaN,NaN,0,0,NaN,0,0,NaN,0,0


The rollup that orders the participant grid. `risk_score` is **null, not zero**, for anyone thestudy is not currently asking anything of (pre-enrollment, complete, withdrawn). `risk_components`stays a dict per cell — it holds each weighted term's point contribution. A rate is suppressed(null) below `RATE_MIN_PARTICIPANTS` / `RATE_MIN_UNITS`, with its `_num` / `_den` still populated.

In [273]:
METRICS_COHORT_FIELDS = (
    "id", "as_of", "phase_filter", "n_participants", "n_active",
    "benchmarks", "series_14d", "integrity", "funnel",
    "decision_points_n", "eligible_n", "sent_n", "delivered_n",
    "cooldown_violations_n", "runin_violations_n", "cap_hit_days", "delivery_failures_n",
)

metrics_cohort_df = frame(
    MetricsCohort.objects.order_by("-as_of"),
    *METRICS_COHORT_FIELDS,
)

cohort_latest = metrics_cohort_df.drop_duplicates("phase_filter")
cohort_latest[["as_of", "phase_filter", "n_participants", "n_active"]]

,as_of,phase_filter,n_participants,n_active
0,2026-09-16 10:00:00+00:00,all,14,11
1,2026-09-16 10:00:00+00:00,phase1,0,0
2,2026-09-16 10:00:00+00:00,phase2,14,11


Append-only, one row per compute run per `phase_filter` (`all`, `phase1`, `phase2`), pruned after`METRICS_COHORT_RETENTION_DAYS`. `cohort_latest` is the row `GET /api/monitor/cohort?phase=` returns.`benchmarks`, `series_14d`, `integrity` and `funnel` stay JSON dicts per cell — each row is alwaysread whole. Note `runin_violations_n` is plural here and singular (`runin_violation_n`) on`MetricsDaily`.

In [274]:
ALERT_FIELDS = ("id", "user_id", "rule_id", "severity", "fired_at", "resolved_at", "payload")

alerts_df = frame(Alert.objects.order_by("-fired_at"), *ALERT_FIELDS)
alerts_open_df = alerts_df[alerts_df["resolved_at"].isna()]

print("alerts:", len(alerts_df), "| open:", len(alerts_open_df))
if len(alerts_open_df):
    print(alerts_open_df.groupby(["severity", "rule_id"]).size())

alerts: 7 | open: 7
severity  rule_id           
critical  cap_exceeded          1
          cooldown_violation    1
          runin_violation       1
          sync_stale            1
high      no_ema_48h            1
          wear_low              1
warning   slot_coverage_low     1
dtype: int64


Includes resolved rows; `alerts_open_df` is what `GET /api/monitor/alerts` serves. **A null`user_id` is a cohort-scoped alert, not missing data** — an alert about the system(`runin_violation`, `sync_stale` with no writer) fires once for the cohort rather than once perparticipant. Severity is a three-tier ladder: sort with `Alert.SEVERITY_RANK`, not on the column,which puts `critical` before `warning` only by accident of spelling.

In [275]:
derived_summary = pd.DataFrame([
    shape("dashboard_metricsdaily", metrics_daily_df),
    shape("dashboard_metricsparticipant", metrics_participant_df),
    shape("dashboard_metricscohort", metrics_cohort_df),
    shape("dashboard_alert", alerts_df),
])
derived_summary

,table,rows,columns
0,dashboard_metricsdaily,256,39
1,dashboard_metricsparticipant,14,23
2,dashboard_metricscohort,3,17
3,dashboard_alert,7,7


All zero rows means the metrics have never been computed against this database. Fill them with`python manage.py recompute_metrics --all` from `backend/` — that writes, so it is not run here.

## Raw source tables (`app_*`)

The ten tables the `dashboard/data/*` modules actually read, taken from the`from app.models import` blocks in `daily.py`, `timeline.py`, `cohort.py`, `participant.py`,`alerts.py` and `views.py`. `StressSample` and `EventDay` are excluded — nothing under`dashboard/` imports either.Field lists are explicit so a later migration adding a column cannot silently change a frame'sshape.

In [276]:
from app.models import (
    EMA,
    CheckinReminder,
    EMAItemResponse,
    EngagementLog,
    HeartRateSample,
    JITAILog,
    PhoneTelemetry,
    User,
    WearableDevice,
    WearableSync,
)

users_df = frame(
    User.objects.order_by("user_id"),
    "user_id", "is_enrolled", "enrolled_at", "gender", "birthdate",
)
if not SYNTHETIC_DATA:
    users_df["has_push_token"] = [
        bool(t) for t in User.objects.order_by("user_id").values_list("push_token", flat=True)
    ]

print("users:", len(users_df), "| enrolled:", int(users_df["is_enrolled"].sum()))
users_df.head()

users: 14 | enrolled: 13


,user_id,is_enrolled,enrolled_at,gender,birthdate
0,1001,True,2026-08-13 14:00:00+00:00,female,2008-06-24
1,1002,False,2026-08-26 14:00:00+00:00,female,2007-07-11
2,1003,True,2026-08-29 14:00:00+00:00,female,2005-07-15
3,1004,True,2026-09-10 14:00:00+00:00,female,2007-03-10
4,1005,True,2026-09-10 14:00:00+00:00,other,2007-02-18


`email`, `first_name`, `last_name` and `password` are left out — they are direct identifiers andnothing in the monitoring layer reads them. The push token is reduced to a boolean for the samereason; its presence is the only part the pipeline cares about (a missing token is why a promptsilently fails to deliver).

In [277]:
wearable_devices_df = frame(
    WearableDevice.objects.order_by("user_id"),
    "id", "user_id", "labfront_participant_id", "is_active", "last_synced_at",
)

wearable_sync_df = frame(
    WearableSync.objects.order_by("user_id", "observed_at"),
    "id", "user_id", "observed_at", "source", "last_synced_at", "samples_written",
)

print("devices:", len(wearable_devices_df), "| sync events:", len(wearable_sync_df))
wearable_devices_df.head()

devices: 14 | sync events: 562


,id,user_id,labfront_participant_id,is_active,last_synced_at
0,1,1001,SYN-000,True,2026-09-15 14:00:00+00:00
1,2,1002,SYN-001,False,2026-09-15 11:00:00+00:00
2,3,1003,SYN-002,False,2026-09-15 01:00:00+00:00
3,4,1004,SYN-003,True,2026-09-15 11:00:00+00:00
4,5,1005,SYN-004,True,2026-09-16 05:00:00+00:00


`WearableSync` is an append-only log of a device's sync clock advancing — it exists because`WearableDevice.last_synced_at` is a single mutable column, so without it a sync outage cannot betold apart from genuine non-wear. **Expect it to be empty**: the mobile client is the intendedwriter but has no wearable code, and `ingest_wearable_data` is still a stub. The monitoring layerreports that as unmeasurable rather than turning everyone critical 72 hours after enrolling.

In [278]:
ema_df = frame(
    EMA.objects.order_by("user_id", "sent_at"),
    "id", "user_id", "prompt_id", "ema_type", "status",
    "sent_at", "responded_at", "expires_at",
    "outcome_window_start", "outcome_window_end",
    "source_jitai_log_id", "served_sub_item_ids",
    "mood", "stress", "energy",
)

ema_item_responses_df = frame(
    EMAItemResponse.objects.order_by("ema_id", "item_id", "sub_item_id"),
    "id", "ema_id", "item_id", "sub_item_id", "response_type",
    "value_numeric", "value_choice", "value_choices",
)

print("ema:", len(ema_df), "| item responses:", len(ema_item_responses_df))
if len(ema_df):
    print(ema_df.groupby(["ema_type", "status"]).size())

ema: 837 | item responses: 15409
ema_type            status   
post_prompt         completed     43
prompt_feedback     completed     43
scheduled_check_in  completed    751
dtype: int64


`served_sub_item_ids` records what a check-in actually put on screen, which is what makes itemcompleteness measurable at all; where it is null, `daily.py` reconstructs the askable set againstthe frozen item bank (`dashboard/data/item_bank.py`). No free-text field is stored anywhere inthese tables — that is an IRB constraint, not an omission here.

In [279]:
checkin_reminders_df = frame(
    CheckinReminder.objects.order_by("user_id", "sent_at"),
    "id", "user_id", "sent_at", "daily_count_at_send",
)

print("reminders:", len(checkin_reminders_df))
checkin_reminders_df.head()

reminders: 600


,id,user_id,sent_at,daily_count_at_send
0,1,1001,2026-08-13 19:30:00+00:00,3
1,2,1001,2026-08-13 21:30:00+00:00,4
2,3,1001,2026-08-13 23:30:00+00:00,5
3,4,1001,2026-08-14 13:30:00+00:00,0
4,5,1001,2026-08-14 15:30:00+00:00,1


The reminder log is what lets a missed check-in slot be classified: `slots_covered` and`slots_reminded_uncovered` are disjoint, and `slots_silent` — reminded never sent at all — is ascheduler failure, not non-compliance.

In [280]:
JITAI_FIELDS = (
    "id", "user_id", "prompt_id", "triggered_at", "trigger_reason", "trigger_signal",
    "decision_point_id", "ema_id",
    "observed_mssd", "threshold_at_decision", "threshold_source",
    "randomization_probability", "randomization_draw",
    "message_arm", "arm_randomization_probability", "arm_randomization_draw",
    "send_prompt", "status",
    "hr_at_trigger", "stress_at_trigger", "ema_mood", "ema_stress", "ema_energy",
    "decision_made_at", "push_sent_at", "device_received_at", "receipt_reported_at",
    "receipt_event_id", "delivery_status", "delivery_error",
    "receipt_platform", "receipt_app_state",
    "eligible_prompt_ids", "evaluated_items", "matched_categories",
    "category_drawn", "fallback_reason",
)

jitai_log_df = frame(JITAILog.objects.order_by("user_id", "triggered_at"), *JITAI_FIELDS)

print("decision points:", len(jitai_log_df))
if len(jitai_log_df):
    print("sent:", int(jitai_log_df["send_prompt"].sum()))
    print(jitai_log_df["threshold_source"].value_counts(dropna=False))

decision points: 428
sent: 73
threshold_source
engine           288
reconstructed    140
Name: count, dtype: int64


Every decision point, sent or not — `send_prompt` is the gate, so the full table is the decisiondenominator and the `send_prompt=True` subset is the dose.`threshold_at_decision` records what `observed_mssd` was actually compared against.`threshold_source` is `engine` when the live decision wrote it and `reconstructed` when`manage.py backfill_thresholds` replayed it; the dashboard's "should have been eligible" markerfires only on `engine` rows. `prompt_id` is a template reference — the message text is neverstored.

In [281]:
engagement_log_df = frame(
    EngagementLog.objects.order_by("user_id", "occurred_at"),
    "id", "user_id", "jitai_log_id", "event_type", "occurred_at", "recorded_at",
)

phone_telemetry_df = frame(
    PhoneTelemetry.objects.order_by("user_id", "occurred_at"),
    "id", "user_id", "session_id", "event_type", "occurred_at", "recorded_at",
    "screen_name", "latency_ms", "metadata",
)

print("engagement:", len(engagement_log_df), "| phone telemetry:", len(phone_telemetry_df))
if len(engagement_log_df):
    print(engagement_log_df["event_type"].value_counts())

engagement: 75 | phone telemetry: 322
event_type
notification_tapped       39
ema_opened                21
notification_dismissed    15
Name: count, dtype: int64


`EngagementLog` backs the opened / acted / dismissed / outcome-captured counts.`PhoneTelemetry` carries `occurred_at` (device clock) alongside `recorded_at` (server clock);their difference is `clock_skew_p95_ms` on `MetricsDaily`, which is **signed** — a device runningahead of the server is a real diagnostic condition, not an error.

In [282]:
hr_qs = HeartRateSample.objects.order_by("user_id", "timestamp")

if SYNTHETIC_DATA:
    heart_rate_df = frame(hr_qs, "id", "user_id", "timestamp", "bpm", "source")
    hr_total = len(heart_rate_df)
    if HR_DAYS is not None:
        cutoff = pd.Timestamp.utcnow() - pd.Timedelta(days=HR_DAYS)
        heart_rate_df = heart_rate_df[heart_rate_df["timestamp"] >= cutoff]
else:
    hr_total = HeartRateSample.objects.count()
    if HR_DAYS is not None:
        hr_qs = hr_qs.filter(timestamp__gte=timezone.now() - timedelta(days=HR_DAYS))
    heart_rate_df = frame(hr_qs, "id", "user_id", "timestamp", "bpm", "source")

print(f"heart rate rows in table: {hr_total}")
print(f"loaded (HR_DAYS={HR_DAYS}): {len(heart_rate_df)}")

heart rate rows in table: 127469
loaded (HR_DAYS=14): 127469


The one table that can run to millions of rows — HR ingests at a 15-second cadence, roughly 720rows per participant per 3-hour session. `HR_DAYS` bounds it to a trailing window; set it to`None` to load the whole table, deliberately. The full count is printed alongside so thetruncation is always visible.`dashboard/data/daily.py` derives wear from this: `wear_valid_pct` is `(840 - gaps>2h) / 840`against the 08:00–22:00 Eastern waking window, `hr_minutes_valid / 840` is minute-level coverage,and `wear_gap_pct` is the complementary long-gap fraction — not what either wear tile shows.

## 6 — Summary

In [283]:
loaded = [
    ("dashboard_metricsdaily", metrics_daily_df),
    ("dashboard_metricsparticipant", metrics_participant_df),
    ("dashboard_metricscohort", metrics_cohort_df),
    ("dashboard_alert", alerts_df),
    ("app_user", users_df),
    ("app_wearabledevice", wearable_devices_df),
    ("app_wearablesync", wearable_sync_df),
    ("app_ema", ema_df),
    ("app_emaitemresponse", ema_item_responses_df),
    ("app_checkinreminder", checkin_reminders_df),
    ("app_jitailog", jitai_log_df),
    ("app_engagementlog", engagement_log_df),
    ("app_phonetelemetry", phone_telemetry_df),
    (f"app_heartratesample (last {HR_DAYS}d)", heart_rate_df),
]

pd.DataFrame([shape(name, df) for name, df in loaded])

,table,rows,columns
0,dashboard_metricsdaily,256,39
1,dashboard_metricsparticipant,14,23
2,dashboard_metricscohort,3,17
3,dashboard_alert,7,7
4,app_user,14,5
5,app_wearabledevice,14,5
6,app_wearablesync,562,6
7,app_ema,837,15
8,app_emaitemresponse,15409,8
9,app_checkinreminder,600,4


In [284]:
# STAGE 1 — COHORT BOARD
import math
from datetime import date, timedelta

import numpy as np

from dashboard.data.cohort import (
    INELIGIBLE_REASONS,
    WAKING_MINUTES,
    WEAR_DIVERGENCE,
    ks_uniform,
    suppress_rate,
    wilson_interval,
)
from dashboard.data.config import (
    NOTIFICATION_WINDOW_END_HOUR,
    NOTIFICATION_WINDOW_START_HOUR,
    PHASE1_USER_IDS,
    SCHEDULED_CHECK_IN_DAILY_CAP,
    UNMEASURABLE_BENCHMARKS,
    WEAR_GAP_MIN,
    randomization_p,
)
from dashboard.data.daily import ACTED_CHOICES, FEEDBACK_TYPE, OUTCOME_TYPES, SCHEDULED_TYPE
from dashboard.data.windows import today_local

SERIES_DAYS = 14
ACTIVE_RETENTION_DAYS = 7
SLOT_HOURS = (NOTIFICATION_WINDOW_END_HOUR - NOTIFICATION_WINDOW_START_HOUR) / SCHEDULED_CHECK_IN_DAILY_CAP
ELIGIBILITY_BAND = (0.10, 0.35)
CAP_HIT_ALARM = 0.10
OUTCOME_CAPTURE_ALARM = 0.60
KS_ALARM_P = 0.01
OPENED_EVENT = "notification_tapped"
SIGNAL_SUB_ITEM = "C0_behavior_change"

In [285]:
def to_local(series):
    return pd.to_datetime(series, utc=True).dt.tz_convert(PARTICIPANT_TZ)


def local_date_of(moment):
    if moment is None or pd.isna(moment):
        return None
    return pd.Timestamp(moment).tz_convert(PARTICIPANT_TZ).date()


def cohort_users(phase="all"):
    ids = set(users_df.loc[users_df["enrolled_at"].notna(), "user_id"])
    if phase == "phase1":
        ids &= set(PHASE1_USER_IDS)
    elif phase == "phase2":
        ids -= set(PHASE1_USER_IDS)
    return sorted(ids)


def withdrawal_dates():
    if metrics_participant_df.empty:
        return {}
    return {
        uid: local_date_of(moment)
        for uid, moment in zip(
            metrics_participant_df["user_id"],
            metrics_participant_df["first_seen_not_enrolled_at"],
        )
        if pd.notna(moment)
    }


def day1_dates():
    enrolled = users_df[users_df["enrolled_at"].notna()]
    return {uid: local_date_of(m) for uid, m in zip(enrolled["user_id"], enrolled["enrolled_at"])}


def participant_days(phase="all", as_of=None):
    as_of = as_of or today_local()
    day1 = day1_dates()
    withdrawn = withdrawal_dates()
    rows = []
    for uid in cohort_users(phase):
        anchor = day1.get(uid)
        if anchor is None:
            continue
        stop = withdrawn.get(uid)
        for offset in range(STUDY_DAYS):
            local = anchor + timedelta(days=offset)
            if local > as_of:
                break
            if stop is not None and local >= stop:
                break
            rows.append({
                "user_id": uid,
                "local_date": local,
                "study_day": offset,
                "is_run_in": offset < RUN_IN_DAYS,
            })
    frame = pd.DataFrame(rows, columns=["user_id", "local_date", "study_day", "is_run_in"])
    return frame.astype({"user_id": "int64", "study_day": "int64", "is_run_in": "bool"})


def benchmark(name, k, n, participants, unit, extra=None):
    if name in UNMEASURABLE_BENCHMARKS:
        return {
            "measurable": False, "suppressed": False, "unit": unit,
            "target": BENCHMARKS.get(name), "value": None,
            "wilson_low": None, "wilson_high": None,
            "numerator": None, "denominator": None, "participants": 0,
        }
    value = suppress_rate(k, n, participants)
    low, high = wilson_interval(k, n) if value is not None else (None, None)
    entry = {
        "measurable": True, "suppressed": value is None, "unit": unit,
        "target": BENCHMARKS.get(name), "value": value,
        "wilson_low": low, "wilson_high": high,
        "numerator": k, "denominator": n, "participants": participants,
    }
    if extra:
        entry.update(extra)
    return entry


def gauge(name, k, n, target=None, alarm=False, extra=None):
    low, high = wilson_interval(k, n)
    entry = {
        "gauge": name, "value": (k / n) if n else None,
        "numerator": k, "denominator": n,
        "wilson_low": low, "wilson_high": high, "target": target,
        "alarm": bool(alarm), "contradiction": bool(n and k > n),
    }
    if extra:
        entry.update(extra)
    return entry

In [286]:
def scheduled_slots(users=None):
    rows = ema_df[ema_df["ema_type"].eq(SCHEDULED_TYPE) & ema_df["status"].eq("completed")].copy()
    if users is not None:
        rows = rows[rows["user_id"].isin(users)]
    if rows.empty:
        return pd.DataFrame(columns=["user_id", "local_date", "slot_index", "hour"])
    local = to_local(rows["sent_at"])
    rows["local_date"] = local.dt.date
    rows["hour"] = local.dt.hour
    rows = rows[
        rows["hour"].ge(NOTIFICATION_WINDOW_START_HOUR)
        & rows["hour"].lt(NOTIFICATION_WINDOW_END_HOUR)
    ]
    rows["slot_index"] = np.floor((rows["hour"] - NOTIFICATION_WINDOW_START_HOUR) / SLOT_HOURS).astype(int)
    return rows[["user_id", "local_date", "slot_index", "hour"]]


def slot_coverage_days(phase="all"):
    days = participant_days(phase)
    slots = scheduled_slots(set(days["user_id"]))
    covered = (
        slots.groupby(["user_id", "local_date"])["slot_index"].nunique().reset_index(name="slots_covered")
        if not slots.empty
        else pd.DataFrame(columns=["user_id", "local_date", "slots_covered"])
    )
    days = days.merge(covered, on=["user_id", "local_date"], how="left")
    days["slots_covered"] = days["slots_covered"].fillna(0).astype(int)
    days["slots_expected"] = SCHEDULED_CHECK_IN_DAILY_CAP
    return days


def slot_coverage(phase="all"):
    days = slot_coverage_days(phase)
    k = int(days["slots_covered"].sum())
    n = int(days["slots_expected"].sum())
    participants = days["user_id"].nunique()
    per_user = days.groupby("user_id")[["slots_covered", "slots_expected"]].sum()
    means = (per_user["slots_covered"] / per_user["slots_expected"])
    mean = float(means.mean()) if len(means) else None
    return benchmark(
        "slot_coverage", k, n, participants, "check-in slots covered",
        extra={
            "label": "Slot coverage (proxy)",
            "proxy": True,
            "participant_mean": mean if suppress_rate(k, n, participants) is not None else None,
            "participant_days": len(days),
        },
    )


def slot_diagnostic(phase="all"):
    users = cohort_users(phase)
    reminders = checkin_reminders_df[checkin_reminders_df["user_id"].isin(users)].copy()
    if reminders.empty:
        return pd.DataFrame(columns=["daily_count_at_send", "local_hour", "n"])
    local = to_local(reminders["sent_at"])
    reminders["local_hour"] = local.dt.hour
    return (
        reminders.groupby(["daily_count_at_send", "local_hour"])
        .size().reset_index(name="n")
        .sort_values(["daily_count_at_send", "local_hour"])
    )

In [287]:
def mrt_day_keys(phase="all"):
    days = participant_days(phase)
    mrt = days[~days["is_run_in"]]
    return set(zip(mrt["user_id"], mrt["local_date"])), mrt


def decisions(phase="all", mrt_only=True):
    users = cohort_users(phase)
    rows = jitai_log_df[jitai_log_df["user_id"].isin(users)].copy()
    anchor = rows["push_sent_at"].fillna(rows["decision_made_at"]).fillna(rows["triggered_at"])
    rows["local_date"] = to_local(anchor).dt.date
    if mrt_only:
        keys, _ = mrt_day_keys(phase)
        mask = np.array([(u, d) in keys for u, d in zip(rows["user_id"], rows["local_date"])], dtype=bool)
        rows = rows[mask]
    return rows


def responded_logs(delivered):
    window = pd.Timedelta(hours=OUTCOME_WINDOW_HOURS)
    push = dict(zip(delivered["id"], pd.to_datetime(delivered["push_sent_at"], utc=True)))
    events = engagement_log_df[
        engagement_log_df["jitai_log_id"].isin(push) & engagement_log_df["event_type"].eq(OPENED_EVENT)
    ]
    responded = set()
    for log_id, occurred in zip(events["jitai_log_id"], pd.to_datetime(events["occurred_at"], utc=True)):
        deadline = push.get(log_id)
        if deadline is not None and pd.notna(deadline) and occurred <= deadline + window:
            responded.add(log_id)
    acted_emas = set(
        ema_item_responses_df.loc[
            ema_item_responses_df["sub_item_id"].eq(SIGNAL_SUB_ITEM)
            & ema_item_responses_df["value_choice"].isin(ACTED_CHOICES),
            "ema_id",
        ]
    )
    feedback = ema_df[
        ema_df["source_jitai_log_id"].isin(push)
        & ema_df["ema_type"].eq(FEEDBACK_TYPE)
        & ema_df["status"].eq("completed")
        & ema_df["id"].isin(acted_emas)
    ]
    responded |= set(feedback["source_jitai_log_id"])
    return responded


def prompt_response(phase="all"):
    rows = decisions(phase)
    sent = rows[rows["send_prompt"].astype(bool)] if not rows.empty else rows
    delivered = sent[sent["device_received_at"].notna()] if not sent.empty else sent
    if delivered.empty:
        return benchmark("prompt_response", 0, 0, 0, "delivered prompts responded to",
                         extra={"window": "weeks 2-5", "by_platform": {}})
    responded = responded_logs(delivered)
    k = int(delivered["id"].isin(responded).sum())
    n = int(len(delivered))
    participants = int(delivered["user_id"].nunique())
    by_platform = {}
    for platform, group in delivered.groupby(delivered["receipt_platform"].fillna("unknown")):
        hits = int(group["id"].isin(responded).sum())
        by_platform[platform] = {
            "numerator": hits,
            "denominator": int(len(group)),
            "participants": int(group["user_id"].nunique()),
            "value": suppress_rate(hits, len(group), group["user_id"].nunique()),
        }
    return benchmark(
        "prompt_response", k, n, participants, "delivered prompts responded to",
        extra={"window": "weeks 2-5", "by_platform": by_platform},
    )

In [288]:
def long_gap_minutes(valid, threshold=WEAR_GAP_MIN):
    total = run = 0
    for flag in valid:
        if flag:
            if run > threshold:
                total += run
            run = 0
        else:
            run += 1
    if run > threshold:
        total += run
    return total


def wear_days(phase="all"):
    days = participant_days(phase)
    start_minute = WAKING_WINDOW_START_HOUR * 60
    hr = heart_rate_df[heart_rate_df["bpm"].gt(0)].copy() if not heart_rate_df.empty else heart_rate_df
    scored = {}
    if not hr.empty:
        local = to_local(hr["timestamp"])
        hr["local_date"] = local.dt.date
        hr["minute"] = local.dt.hour * 60 + local.dt.minute
        hr = hr[hr["minute"].ge(start_minute) & hr["minute"].lt(WAKING_WINDOW_END_HOUR * 60)]
        for (uid, local_date), group in hr.groupby(["user_id", "local_date"]):
            bins = np.zeros(WAKING_MINUTES, dtype=bool)
            bins[group["minute"].to_numpy() - start_minute] = True
            scored[(uid, local_date)] = (float(bins.mean()), long_gap_minutes(bins))
    keys = list(zip(days["user_id"], days["local_date"]))
    days["minute_coverage"] = [scored.get(key, (np.nan, np.nan))[0] for key in keys]
    days["gap_minutes"] = [scored.get(key, (np.nan, np.nan))[1] for key in keys]
    days["gap_coverage"] = (WAKING_MINUTES - days["gap_minutes"]) / WAKING_MINUTES
    return days


def wear_coverage(phase="all"):
    days = wear_days(phase)
    scored = days[days["gap_coverage"].notna()]
    n = int(len(scored))
    k = int((scored["gap_coverage"] >= BENCHMARKS["wear"]).sum())
    participants = int(scored["user_id"].nunique())
    diverged = int((scored["gap_coverage"] - scored["minute_coverage"] >= WEAR_DIVERGENCE).sum())
    return benchmark(
        "wear", k, n, participants, "participant-days meeting the coverage target",
        extra={
            "mean_gap_coverage": float(scored["gap_coverage"].mean()) if n else None,
            "mean_minute_coverage": float(scored["minute_coverage"].mean()) if n else None,
            "burst_charging_days": diverged,
            "divergence_threshold": WEAR_DIVERGENCE,
            "gap_threshold_min": WEAR_GAP_MIN,
            "maps_to_metricsdaily": {
                "gap_coverage": "wear_valid_pct",
                "minute_coverage": "hr_minutes_valid / 840",
            },
        },
    )

In [289]:
def retention(phase="all", as_of=None):
    as_of = as_of or today_local()
    users = cohort_users(phase)
    day1 = day1_dates()
    withdrawn = withdrawal_dates()
    started = [uid for uid in users if day1.get(uid) is not None and day1[uid] <= as_of]
    complete = [uid for uid in started if (as_of - day1[uid]).days >= STUDY_DAYS - 1]
    enrolled_now = set(users_df.loc[users_df["is_enrolled"].astype(bool), "user_id"])
    retained = [uid for uid in complete if uid in enrolled_now and uid not in withdrawn]

    in_window = [
        uid for uid in started
        if (as_of - day1[uid]).days < STUDY_DAYS
        and (withdrawn.get(uid) is None or withdrawn[uid] > as_of)
    ]
    slots = scheduled_slots(set(in_window))
    cutoff = as_of - timedelta(days=ACTIVE_RETENTION_DAYS)
    recent = set(slots.loc[slots["local_date"] > cutoff, "user_id"]) if not slots.empty else set()
    active_k = len([uid for uid in in_window if uid in recent])

    formal = benchmark(
        "retention", len(retained), len(complete), len(complete),
        "participants retained at day 35",
    )
    formal["active"] = benchmark(
        "retention", active_k, len(in_window), len(in_window),
        f"participants with a scheduled check-in in {ACTIVE_RETENTION_DAYS} days",
    )
    formal["started_day1"] = len(started)
    formal["reached_day35"] = len(complete)
    return formal


def hair_sample(phase="all"):
    return {
        "measurable": False, "suppressed": False,
        "unit": "hair samples collected",
        "label": "Hair sample",
        "target": 0.90, "value": None,
        "wilson_low": None, "wilson_high": None,
        "numerator": None, "denominator": None, "participants": 0,
        "source": "external: no table in production; needs an RA sheet loaded into a derived table",
    }

In [290]:
def series_14d(phase="all", as_of=None):
    as_of = as_of or today_local()
    window_start = as_of - timedelta(days=SERIES_DAYS - 1)
    slots = slot_coverage_days(phase)
    wear = wear_days(phase)
    rows_jitai = decisions(phase, mrt_only=False)
    delivered = (
        rows_jitai[rows_jitai["send_prompt"].astype(bool) & rows_jitai["device_received_at"].notna()]
        if not rows_jitai.empty else rows_jitai
    )
    responded = responded_logs(delivered) if not delivered.empty else set()

    out = []
    for offset in range(SERIES_DAYS):
        local = window_start + timedelta(days=offset)
        day_slots = slots[slots["local_date"].eq(local)] if not slots.empty else slots
        day_wear = wear[wear["local_date"].eq(local) & wear["gap_coverage"].notna()] if not wear.empty else wear
        day_delivered = delivered[delivered["local_date"].eq(local)] if not delivered.empty else delivered
        participants = int(day_slots["user_id"].nunique()) if len(day_slots) else 0

        covered = int(day_slots["slots_covered"].sum()) if len(day_slots) else 0
        expected = int(day_slots["slots_expected"].sum()) if len(day_slots) else 0
        wear_scored = int(len(day_wear))
        wear_met = int((day_wear["gap_coverage"] >= BENCHMARKS["wear"]).sum()) if wear_scored else 0
        delivered_n = int(len(day_delivered))
        responded_n = int(day_delivered["id"].isin(responded).sum()) if delivered_n else 0

        out.append({
            "date": local,
            "participants": participants,
            "slots_covered": covered,
            "slots_expected": expected,
            "slot_coverage": suppress_rate(covered, expected, participants),
            "delivered_n": delivered_n,
            "responded_n": responded_n,
            "prompt_response": suppress_rate(
                responded_n, delivered_n, int(day_delivered["user_id"].nunique()) if delivered_n else 0),
            "wear_days_scored": wear_scored,
            "wear_days_met": wear_met,
            "wear_pass_rate": suppress_rate(
                wear_met, wear_scored, int(day_wear["user_id"].nunique()) if wear_scored else 0),
        })
    return pd.DataFrame(out)

In [291]:
def enrollment_funnel(phase="all", as_of=None):
    as_of = as_of or today_local()
    users = cohort_users(phase)
    day1 = day1_dates()
    withdrawn = withdrawal_dates()
    started = [uid for uid in users if day1.get(uid) is not None and day1[uid] <= as_of]
    active = [
        uid for uid in started
        if (as_of - day1[uid]).days < STUDY_DAYS
        and (withdrawn.get(uid) is None or withdrawn[uid] > as_of)
    ]
    completed = [uid for uid in started if (as_of - day1[uid]).days >= STUDY_DAYS - 1]
    return pd.DataFrame([
        {"stage": "consented", "n": None, "measurable": False,
         "detail": "nothing in the schema records consent"},
        {"stage": "started day 1", "n": len(started), "measurable": True, "detail": None},
        {"stage": "active today", "n": len(active), "measurable": True, "detail": None},
        {"stage": "completed day 34", "n": len(completed), "measurable": True, "detail": None},
        {"stage": "withdrew", "n": len([u for u in started if u in withdrawn]), "measurable": True,
         "detail": "first_seen_not_enrolled_at, resolution = polling interval"},
    ])

In [292]:
def cooldown_violations(rows):
    violations = 0
    pairs = []
    sent = rows[rows["send_prompt"].astype(bool)] if not rows.empty else rows
    if sent.empty:
        return 0, pairs
    anchor = pd.to_datetime(sent["push_sent_at"].fillna(sent["decision_made_at"]), utc=True)
    frame = pd.DataFrame({"user_id": sent["user_id"].to_numpy(), "id": sent["id"].to_numpy(), "at": anchor.to_numpy()})
    for uid, group in frame.sort_values("at").groupby("user_id"):
        times = list(group["at"])
        ids = list(group["id"])
        for i in range(1, len(times)):
            minutes = (times[i] - times[i - 1]).total_seconds() / 60
            if minutes < JITAI_COOLDOWN_MINUTES:
                violations += 1
                pairs.append({"user_id": uid, "earlier_id": ids[i - 1], "later_id": ids[i],
                              "gap_min": round(minutes, 1)})
    return violations, pairs


def outcome_capture(delivered):
    if delivered.empty:
        return 0
    linked = ema_df[
        ema_df["source_jitai_log_id"].isin(set(delivered["id"]))
        & ema_df["ema_type"].isin(OUTCOME_TYPES)
        & ema_df["responded_at"].notna()
    ]
    if linked.empty:
        return 0
    responded_at = pd.to_datetime(linked["responded_at"], utc=True)
    start = pd.to_datetime(linked["outcome_window_start"], utc=True)
    end = pd.to_datetime(linked["outcome_window_end"], utc=True)
    inside = linked[(responded_at >= start) & (responded_at <= end)]
    return int(inside["source_jitai_log_id"].nunique())


def mrt_integrity(phase="all"):
    rows = decisions(phase)
    keys, mrt_days = mrt_day_keys(phase)
    decision_points = int(len(rows))
    eligible = int(rows["randomization_draw"].notna().sum()) if decision_points else 0
    sent = rows[rows["send_prompt"].astype(bool)] if decision_points else rows
    delivered = sent[sent["device_received_at"].notna()] if len(sent) else sent

    ineligible_with_draw = int(
        (rows["randomization_draw"].notna() & rows["trigger_reason"].isin(INELIGIBLE_REASONS)).sum()
    ) if decision_points else 0
    eligible_no_draw = int(
        (rows["randomization_draw"].isna() & ~rows["trigger_reason"].isin(INELIGIBLE_REASONS)).sum()
    ) if decision_points else 0

    drawn = rows[rows["randomization_draw"].notna()] if decision_points else rows
    draws = drawn["randomization_draw"].astype(float).tolist() if len(drawn) else []
    probabilities = drawn["randomization_probability"].astype(float).tolist() if len(drawn) else []
    flags = drawn["send_prompt"].astype(bool).tolist() if len(drawn) else []
    mismatches = sum(1 for flag, draw, p in zip(flags, draws, probabilities) if flag != (draw < p))
    ks_stat, ks_p = ks_uniform(draws)

    cap_days = int(rows.loc[rows["trigger_reason"].eq("daily cap reached")]
                   .drop_duplicates(["user_id", "local_date"]).shape[0]) if decision_points else 0
    violations, violation_pairs = cooldown_violations(rows)
    captured = outcome_capture(delivered)

    scheduled_emas = ema_df[
        ema_df["ema_type"].eq(SCHEDULED_TYPE) & ema_df["status"].eq("completed")
        & ema_df["user_id"].isin(set(mrt_days["user_id"]))
    ]
    scheduled_n = 0
    if not scheduled_emas.empty:
        local = to_local(scheduled_emas["sent_at"]).dt.date
        scheduled_n = int(sum((u, d) in keys for u, d in zip(scheduled_emas["user_id"], local)))

    send_target = randomization_p()
    send = gauge("send_rate", int(len(sent)), eligible, target=send_target)
    send["alarm"] = bool(
        send["wilson_low"] is not None
        and not (send["wilson_low"] <= send_target <= send["wilson_high"])
    )
    eligibility = gauge("eligibility_rate", eligible, decision_points, target=ELIGIBILITY_BAND)
    eligibility["alarm"] = bool(
        eligibility["value"] is not None
        and not (ELIGIBILITY_BAND[0] <= eligibility["value"] <= ELIGIBILITY_BAND[1])
    )
    eligibility["availability_confound"] = (decision_points / scheduled_n) if scheduled_n else None
    eligibility["ema_scheduled_n"] = scheduled_n
    eligibility["reason_ineligible_with_draw"] = ineligible_with_draw
    eligibility["reason_eligible_but_no_draw"] = eligible_no_draw
    eligibility["alarm_reason_disagreement"] = bool(ineligible_with_draw or eligible_no_draw)

    cap = gauge("cap_hit_rate", cap_days, int(len(mrt_days)), target=CAP_HIT_ALARM)
    cap["alarm"] = bool(cap["value"] is not None and cap["value"] > CAP_HIT_ALARM)
    outcome = gauge("outcome_capture", captured, int(len(delivered)), target=OUTCOME_CAPTURE_ALARM)
    outcome["alarm"] = bool(outcome["value"] is not None and outcome["value"] < OUTCOME_CAPTURE_ALARM)

    return {
        "window": "weeks 2-5 (study_day >= run-in)",
        "eligibility_rate": eligibility,
        "send_rate": send,
        "randomization_audit": {
            "gauge": "randomization_audit",
            "draws": len(draws),
            "mismatches": mismatches,
            "ks_statistic": ks_stat,
            "ks_p_value": ks_p,
            "alarm_p": KS_ALARM_P,
            "alarm": bool(mismatches or (ks_p is not None and ks_p < KS_ALARM_P)),
        },
        "cap_hit_rate": cap,
        "cooldown": {
            "gauge": "cooldown",
            "violations": violations,
            "threshold_min": JITAI_COOLDOWN_MINUTES,
            "alarm": bool(violations),
            "pairs": violation_pairs,
        },
        "outcome_capture": outcome,
        "decision_points_n": decision_points,
        "eligible_n": eligible,
        "sent_n": int(len(sent)),
        "delivered_n": int(len(delivered)),
    }

In [293]:
def alert_feed(include_resolved=False):
    rows = alerts_df if include_resolved else alerts_open_df
    if rows.empty:
        return pd.DataFrame(columns=["fired_at", "severity", "rule_id", "user_id", "scope", "link_date", "payload"])
    feed = rows.copy()
    feed["scope"] = np.where(feed["user_id"].isna(), "cohort", "participant")
    feed["link_date"] = [
        (payload or {}).get("date") or (payload or {}).get("local_date")
        for payload in feed["payload"]
    ]
    feed["rank"] = feed["severity"].map(Alert.SEVERITY_RANK)
    feed = feed.sort_values(["rank", "fired_at"], ascending=[True, False])
    return feed[["fired_at", "severity", "rule_id", "user_id", "scope", "link_date", "resolved_at", "payload"]]

In [294]:
def cohort_board(phase="all", as_of=None):
    return {
        "phase": phase,
        "as_of": as_of or today_local(),
        "n_participants": len(cohort_users(phase)),
        "benchmarks": {
            "slot_coverage": slot_coverage(phase),
            "prompt_response": prompt_response(phase),
            "wear": wear_coverage(phase),
            "retention": retention(phase, as_of),
            "hair_sample": hair_sample(phase),
        },
        "series_14d": series_14d(phase, as_of),
        "funnel": enrollment_funnel(phase, as_of),
        "integrity": mrt_integrity(phase),
        "alerts": alert_feed(),
    }


def benchmark_table(board):
    rows = []
    for name, entry in board["benchmarks"].items():
        rows.append({
            "tile": entry.get("label", name),
            "measurable": entry["measurable"],
            "suppressed": entry["suppressed"],
            "value": entry["value"],
            "target": entry["target"],
            "wilson_low": entry["wilson_low"],
            "wilson_high": entry["wilson_high"],
            "numerator": entry["numerator"],
            "denominator": entry["denominator"],
            "participants": entry["participants"],
            "unit": entry["unit"],
        })
    return pd.DataFrame(rows)


def integrity_table(board):
    rows = []
    for key, entry in board["integrity"].items():
        if not isinstance(entry, dict):
            continue
        rows.append({
            "gauge": entry.get("gauge", key),
            "value": entry.get("value"),
            "numerator": entry.get("numerator"),
            "denominator": entry.get("denominator"),
            "target": entry.get("target"),
            "alarm": entry.get("alarm"),
            "contradiction": entry.get("contradiction"),
        })
    return pd.DataFrame(rows)


board = cohort_board("all")
print("phase:", board["phase"], "| as_of:", board["as_of"], "| participants:", board["n_participants"])

phase: all | as_of: 2026-09-16 | participants: 14


In [295]:
benchmark_table(board)

,tile,measurable,suppressed,value,target,wilson_low,wilson_high,numerator,denominator,participants,unit
0,Slot coverage (proxy),True,False,0.488932,0.75,0.463992,0.513927,751.0,1536.0,14,check-in slots covered
1,prompt_response,True,False,0.769231,0.70,0.638662,0.862757,40.0,52.0,10,delivered prompts responded to
2,wear,True,False,0.981928,0.80,0.948220,0.993835,163.0,166.0,14,participant-days meeting the coverage target
3,retention,True,True,NaN,0.85,NaN,NaN,1.0,1.0,1,participants retained at day 35
4,Hair sample,False,False,NaN,0.90,NaN,NaN,NaN,NaN,0,hair samples collected


In [296]:
integrity_table(board)

,gauge,value,numerator,denominator,target,alarm,contradiction
0,eligibility_rate,0.299505,121.0,404.0,"(0.1, 0.35)",False,False
1,send_rate,0.504132,61.0,121.0,0.5,False,False
2,randomization_audit,NaN,NaN,NaN,None,False,None
3,cap_hit_rate,0.018868,3.0,159.0,0.1,False,False
4,cooldown,NaN,NaN,NaN,None,True,None
5,outcome_capture,0.673077,35.0,52.0,0.6,False,False


In [297]:
board["funnel"]

,stage,n,measurable,detail
0,consented,NaN,False,nothing in the schema records consent
1,started day 1,14.0,True,None
2,active today,13.0,True,None
3,completed day 34,1.0,True,None
4,withdrew,1.0,True,"first_seen_not_enrolled_at, resolution = polli..."


In [298]:
board["series_14d"]

,date,participants,slots_covered,slots_expected,slot_coverage,delivered_n,responded_n,prompt_response,wear_days_scored,wear_days_met,wear_pass_rate
0,2026-09-03,10,31,60,0.516667,1,1,None,10,10,None
1,2026-09-04,11,34,66,0.515152,2,2,None,11,11,None
2,2026-09-05,11,30,66,0.454545,2,1,None,11,11,None
3,2026-09-06,11,26,66,0.393939,3,2,None,11,11,None
4,2026-09-07,11,33,66,0.500000,2,0,None,11,11,None
5,2026-09-08,11,26,66,0.393939,4,4,None,11,11,None
6,2026-09-09,11,35,66,0.530303,4,1,None,11,11,None
7,2026-09-10,12,32,72,0.444444,6,6,None,12,12,None
8,2026-09-11,13,36,78,0.461538,3,2,None,13,13,None
9,2026-09-12,13,38,78,0.487179,7,5,None,13,13,None


In [299]:
board["alerts"]

,fired_at,severity,rule_id,user_id,scope,link_date,resolved_at,payload
0,2026-09-16 10:00:00+00:00,critical,runin_violation,NaN,cohort,2026-09-15,NaT,"{'detail': 'engine has no run-in gate', 'date'..."
1,2026-09-16 10:00:00+00:00,critical,sync_stale,NaN,cohort,2026-09-15,NaT,"{'measurable': False, 'date': '2026-09-15'}"
2,2026-09-16 10:00:00+00:00,critical,cooldown_violation,1007.0,participant,2026-09-15,NaT,"{'gap_min': 28, 'date': '2026-09-15'}"
3,2026-09-16 10:00:00+00:00,critical,cap_exceeded,1008.0,participant,2026-09-15,NaT,"{'delivered_n': 6, 'date': '2026-09-15'}"
4,2026-09-16 10:00:00+00:00,high,no_ema_48h,1009.0,participant,2026-09-15,NaT,"{'hours': 62, 'date': '2026-09-15'}"
5,2026-09-16 10:00:00+00:00,high,wear_low,1010.0,participant,2026-09-15,NaT,"{'consecutive_days': 3, 'date': '2026-09-15'}"
6,2026-09-16 10:00:00+00:00,warning,slot_coverage_low,1011.0,participant,2026-09-15,NaT,"{'coverage': 0.31, 'date': '2026-09-15'}"


In [300]:
board["benchmarks"]["prompt_response"]["by_platform"]

{'android': {'numerator': 19,
  'denominator': 25,
  'participants': 9,
  'value': None},
 'ios': {'numerator': 21, 'denominator': 27, 'participants': 9, 'value': None}}

In [301]:
slot_diagnostic("all")

,daily_count_at_send,local_hour,n
0,0,9,65
1,1,11,60
2,2,13,84
3,3,15,113
4,4,17,130
5,5,19,148


In [302]:
# STAGE 2 / 3 — ITEM BANK AND COMPLETENESS
from dashboard.data.item_bank import load_item_bank, sub_item_index
from dashboard.data.participant import (
    RISK_COVERAGE_FLOOR,
    RISK_EMA_STALE_CAP,
    RISK_MISSING_SIGNAL_CAP,
    RISK_SYNC_STALE_HOURS,
    RISK_TRAILING_DAYS,
    RISK_WEAR_FLOOR,
    RISK_WEIGHTS,
)
from dashboard.data.windows import scheduled_slot_bounds

MINUTES_PER_DAY = 1440
SIGNAL_SUB_ITEMS = ("B1_valence", "B1_arousal", "B2_stress")
GRID_METRICS = ("slot_coverage", "wear", "delivered_n", "completeness_mean")


def is_answered(row):
    return (
        pd.notna(row.get("value_numeric"))
        or (row.get("value_choice") not in (None, "") and pd.notna(row.get("value_choice")))
        or bool(row.get("value_choices"))
    )


def answered_maps(ema_ids):
    subset = ema_item_responses_df[ema_item_responses_df["ema_id"].isin(set(ema_ids))]
    out = {}
    for record in subset.to_dict("records"):
        entry = out.setdefault(record["ema_id"], {})
        if is_answered(record):
            value = record["value_numeric"]
            if pd.isna(value):
                value = record["value_choice"] if pd.notna(record["value_choice"]) else record["value_choices"]
            entry[record["sub_item_id"]] = {"item_id": record["item_id"], "value": value}
    return out


def satisfies(condition, value):
    if "equals" in condition:
        return value == condition["equals"]
    if "not_equals" in condition:
        return value is not None and value != condition["not_equals"]
    return True


def candidate_sub_items(served, answers, bank):
    index = sub_item_index()
    if served:
        return [(sub_id, index[sub_id]) for sub_id in served if sub_id in index]
    candidates = []
    for item_id in {entry["item_id"] for entry in answers.values()}:
        item = bank.get(item_id)
        if item is None:
            continue
        for sub in item["sub_items"]:
            if "schedule_condition" in sub and sub["sub_item_id"] not in answers:
                continue
            candidates.append((sub["sub_item_id"], sub))
    return candidates


def askable_sub_items(served, answers, bank=None):
    bank = bank or load_item_bank()
    values = {sub_id: entry["value"] for sub_id, entry in answers.items()}
    askable = []
    for sub_id, sub in candidate_sub_items(served, answers, bank):
        depends_on = sub.get("depends_on")
        if depends_on and not satisfies(depends_on, values.get(depends_on["sub_item_id"])):
            continue
        askable.append(sub_id)
    return askable


def ema_completeness(ema_rows):
    bank = load_item_bank()
    maps = answered_maps(ema_rows["id"])
    out = []
    for record in ema_rows.to_dict("records"):
        answers = maps.get(record["id"], {})
        served = record.get("served_sub_item_ids") or None
        askable = askable_sub_items(served, answers, bank)
        answered = [sub_id for sub_id in askable if sub_id in answers]
        out.append({
            "ema_id": record["id"],
            "user_id": record["user_id"],
            "ema_type": record["ema_type"],
            "askable_n": len(askable),
            "answered_n": len(answered),
            "completeness": (len(answered) / len(askable)) if askable else None,
            "missing_b1b2": (
                not all(sub_id in answers for sub_id in SIGNAL_SUB_ITEMS)
                if record["ema_type"] in (SCHEDULED_TYPE,) + tuple(OUTCOME_TYPES) else None
            ),
            "item_bank_version": ITEM_BANK_VERSION,
            "askable": askable,
            "answered": answered,
        })
    return pd.DataFrame(out, columns=[
        "ema_id", "user_id", "ema_type", "askable_n", "answered_n", "completeness",
        "missing_b1b2", "item_bank_version", "askable", "answered",
    ])

In [303]:
# STAGE 2 — PARTICIPANT-DAY BACKBONE
def sync_measurable():
    if not wearable_sync_df.empty:
        return True
    return bool(wearable_devices_df["last_synced_at"].notna().any())


def slot_classification(phase="all"):
    days = participant_days(phase)
    slots = scheduled_slots(set(days["user_id"]))
    covered = {}
    for uid, local_date, slot_index in zip(slots["user_id"], slots["local_date"], slots["slot_index"]):
        covered.setdefault((uid, local_date), set()).add(int(slot_index))

    reminders = checkin_reminders_df[checkin_reminders_df["user_id"].isin(set(days["user_id"]))].copy()
    reminded = {}
    index_reminded = {}
    if not reminders.empty:
        local = to_local(reminders["sent_at"])
        reminders["local_date"] = local.dt.date
        reminders["hour"] = local.dt.hour
        in_window = reminders[
            reminders["hour"].ge(NOTIFICATION_WINDOW_START_HOUR)
            & reminders["hour"].lt(NOTIFICATION_WINDOW_END_HOUR)
        ]
        for uid, local_date, hour in zip(in_window["user_id"], in_window["local_date"], in_window["hour"]):
            slot = int((hour - NOTIFICATION_WINDOW_START_HOUR) // SLOT_HOURS)
            reminded.setdefault((uid, local_date), set()).add(slot)
        for uid, local_date, index in zip(reminders["user_id"], reminders["local_date"], reminders["daily_count_at_send"]):
            if 0 <= index < SCHEDULED_CHECK_IN_DAILY_CAP:
                index_reminded.setdefault((uid, local_date), set()).add(int(index))

    rows = []
    for key in zip(days["user_id"], days["local_date"]):
        hit = covered.get(key, set())
        nudged = reminded.get(key, set()) - hit
        rows.append({
            "slots_covered": len(hit),
            "slots_reminded_uncovered": len(nudged),
            "slots_silent": SCHEDULED_CHECK_IN_DAILY_CAP - len(hit) - len(nudged),
            "slots_reminded_by_index": len(index_reminded.get(key, set()) - hit),
        })
    counts = pd.DataFrame(rows, columns=[
        "slots_covered", "slots_reminded_uncovered", "slots_silent", "slots_reminded_by_index",
    ]).astype("int64")
    return pd.concat([days.reset_index(drop=True), counts], axis=1)


def daily_grid_metrics(phase="all"):
    days = slot_classification(phase)
    wear = wear_days(phase)[["user_id", "local_date", "gap_coverage", "minute_coverage", "gap_minutes"]]
    days = days.merge(wear, on=["user_id", "local_date"], how="left")

    logs = decisions(phase, mrt_only=False)
    if logs.empty:
        days["delivered_n"] = 0
        days["decision_points_n"] = 0
        days["eligible_n"] = 0
        days["sent_n"] = 0
    else:
        counts = logs.assign(
            delivered=logs["device_received_at"].notna().astype(int),
            eligible=logs["randomization_draw"].notna().astype(int),
            sent=logs["send_prompt"].astype(bool).astype(int),
        ).groupby(["user_id", "local_date"]).agg(
            delivered_n=("delivered", "sum"),
            decision_points_n=("delivered", "size"),
            eligible_n=("eligible", "sum"),
            sent_n=("sent", "sum"),
        ).reset_index()
        days = days.merge(counts, on=["user_id", "local_date"], how="left")
        for column in ("delivered_n", "decision_points_n", "eligible_n", "sent_n"):
            days[column] = days[column].fillna(0).astype(int)

    emas = ema_df[
        ema_df["user_id"].isin(set(days["user_id"])) & ema_df["status"].eq("completed")
    ].copy()
    if emas.empty:
        days["completeness_mean"] = np.nan
        days["ema_missing_b1b2_n"] = 0
    else:
        emas["local_date"] = to_local(emas["sent_at"]).dt.date
        scored = ema_completeness(emas[emas["ema_type"].isin((SCHEDULED_TYPE,) + tuple(OUTCOME_TYPES))])
        scored = scored.merge(
            emas[["id", "local_date"]].rename(columns={"id": "ema_id"}), on="ema_id", how="left")
        rollup = scored.groupby(["user_id", "local_date"]).agg(
            completeness_mean=("completeness", "mean"),
            ema_missing_b1b2_n=("missing_b1b2", "sum"),
        ).reset_index()
        days = days.merge(rollup, on=["user_id", "local_date"], how="left")
        days["ema_missing_b1b2_n"] = days["ema_missing_b1b2_n"].fillna(0).astype(int)

    days["slot_coverage"] = days["slots_covered"] / days["slots_expected"] if "slots_expected" in days else np.nan
    days["slots_expected"] = SCHEDULED_CHECK_IN_DAILY_CAP
    days["slot_coverage"] = days["slots_covered"] / SCHEDULED_CHECK_IN_DAILY_CAP
    days["wear"] = days["gap_coverage"]
    return days

In [304]:
# STAGE 2A — HEATMAP
def alert_days_by_user():
    out = {}
    if alerts_open_df.empty:
        return out
    for uid, payload, fired_at in zip(
        alerts_open_df["user_id"], alerts_open_df["payload"], alerts_open_df["fired_at"]
    ):
        if pd.isna(uid):
            continue
        stamp = (payload or {}).get("date") or (payload or {}).get("local_date")
        local = pd.Timestamp(stamp).date() if stamp else local_date_of(fired_at)
        out.setdefault(int(uid), set()).add(local)
    return out


def grid(metric="slot_coverage", phase="all", axis="study_day"):
    if metric not in GRID_METRICS:
        raise ValueError(f"metric must be one of {GRID_METRICS}")
    days = daily_grid_metrics(phase)
    scores = risk_scores(phase).set_index("user_id")
    flagged = alert_days_by_user()

    if axis == "study_day":
        columns = list(range(STUDY_DAYS))
        key_of = lambda row: row["study_day"]
    else:
        present = sorted(set(days["local_date"]))
        columns = present
        key_of = lambda row: row["local_date"]

    rows = []
    for uid in sorted(set(days["user_id"])):
        own = days[days["user_id"].eq(uid)]
        by_key = {key_of(row): row for row in own.to_dict("records")}
        values, run_in, silent, alert, dates = [], [], [], [], []
        for column in columns:
            row = by_key.get(column)
            values.append(None if row is None else (None if pd.isna(row[metric]) else float(row[metric])))
            run_in.append(None if row is None else bool(row["is_run_in"]))
            silent.append(None if row is None else int(row["slots_silent"]) > 0)
            dates.append(None if row is None else row["local_date"])
            alert.append(False if row is None else row["local_date"] in flagged.get(uid, set()))
        rows.append({
            "user_id": uid,
            "risk_score": scores.at[uid, "risk_score"] if uid in scores.index else None,
            "risk_components": scores.at[uid, "risk_components"] if uid in scores.index else None,
            "phase": scores.at[uid, "phase"] if uid in scores.index else None,
            "values": values,
            "run_in": run_in,
            "silent": silent,
            "alert": alert,
            "local_dates": dates,
        })
    rows.sort(key=lambda r: (r["risk_score"] is None, -(r["risk_score"] or 0)))
    return {
        "metric": metric,
        "axis": axis,
        "phase": phase,
        "columns": columns,
        "benchmark": {"slot_coverage": BENCHMARKS["slot_coverage"], "wear": BENCHMARKS["wear"],
                      "delivered_n": DAILY_PROMPT_CAP, "completeness_mean": None}[metric],
        "rows": rows,
    }


def grid_frame(metric="slot_coverage", phase="all", axis="study_day"):
    payload = grid(metric, phase, axis)
    return pd.DataFrame(
        [row["values"] for row in payload["rows"]],
        index=[row["user_id"] for row in payload["rows"]],
        columns=payload["columns"],
    )

In [305]:
# STAGE 2B — SLOT SPLIT
def slot_split(phase="all", days=RISK_TRAILING_DAYS, as_of=None):
    as_of = as_of or today_local()
    frame = daily_grid_metrics(phase)
    cutoff = as_of - timedelta(days=days)
    recent = frame[frame["local_date"].gt(cutoff)]
    if recent.empty:
        return pd.DataFrame(columns=["user_id", "covered", "reminded", "silent", "days"])
    split = recent.groupby("user_id").agg(
        covered=("slots_covered", "sum"),
        reminded=("slots_reminded_uncovered", "sum"),
        silent=("slots_silent", "sum"),
        days=("local_date", "nunique"),
    ).reset_index()
    total = split[["covered", "reminded", "silent"]].sum(axis=1)
    for column in ("covered", "reminded", "silent"):
        split[f"{column}_pct"] = split[column] / total
    return split

In [306]:
# STAGE 2C — RISK SCORE
def phase_of(study_day_now, withdrawn):
    if withdrawn is not None:
        return "withdrawn"
    if study_day_now is None or study_day_now < 0:
        return "pre_enrollment"
    if study_day_now < RUN_IN_DAYS:
        return "run_in"
    if study_day_now < STUDY_DAYS:
        return "mrt"
    return "complete"


def last_sync_ages(as_of=None):
    as_of = as_of or pd.Timestamp.utcnow()
    ages = {}
    for uid, moment in zip(wearable_devices_df["user_id"], wearable_devices_df["last_synced_at"]):
        if pd.notna(moment):
            ages[uid] = (as_of - pd.Timestamp(moment)).total_seconds() / 3600
    return ages


def risk_scores(phase="all", as_of=None):
    as_of = as_of or today_local()
    frame = daily_grid_metrics(phase)
    day1 = day1_dates()
    withdrawn = withdrawal_dates()
    ages = last_sync_ages()
    measurable = sync_measurable()
    slots = scheduled_slots(set(cohort_users(phase)))
    last_ema = slots.groupby("user_id")["local_date"].max().to_dict() if not slots.empty else {}

    out = []
    for uid in cohort_users(phase):
        anchor = day1.get(uid)
        study_day_now = (as_of - anchor).days if anchor else None
        current = phase_of(study_day_now, withdrawn.get(uid))
        if current not in ("run_in", "mrt"):
            out.append({"user_id": uid, "phase": current, "study_day_now": study_day_now,
                        "risk_score": None, "risk_components": None})
            continue

        own = frame[frame["user_id"].eq(uid)]
        trailing = own[own["local_date"].gt(as_of - timedelta(days=RISK_TRAILING_DAYS))]
        stale_days = (as_of - last_ema[uid]).days if uid in last_ema else RISK_EMA_STALE_CAP
        age = ages.get(uid)

        components = {
            "ema_stale": RISK_WEIGHTS["ema_stale"] * min(stale_days, RISK_EMA_STALE_CAP),
            "low_coverage": RISK_WEIGHTS["low_coverage"] * int(
                (trailing["slot_coverage"] < RISK_COVERAGE_FLOOR).sum()),
            "sync_stale": RISK_WEIGHTS["sync_stale"] * int(
                bool(measurable and age is not None and age > RISK_SYNC_STALE_HOURS)),
            "low_wear": RISK_WEIGHTS["low_wear"] * int(
                (trailing["gap_coverage"] < RISK_WEAR_FLOOR).sum()),
            "missing_signal": RISK_WEIGHTS["missing_signal"] * min(
                int(trailing["ema_missing_b1b2_n"].sum()), RISK_MISSING_SIGNAL_CAP),
            "open_critical": RISK_WEIGHTS["open_critical"] * int(bool(
                not alerts_open_df.empty
                and ((alerts_open_df["user_id"] == uid)
                     & alerts_open_df["severity"].isin(Alert.ACTIONABLE_SEVERITIES)).any()
            )),
        }
        out.append({
            "user_id": uid,
            "phase": current,
            "study_day_now": study_day_now,
            "risk_score": int(sum(components.values())),
            "risk_components": components,
            "sync_measurable": measurable,
        })
    scores = pd.DataFrame(out, columns=[
        "user_id", "phase", "study_day_now", "risk_score", "risk_components", "sync_measurable",
    ])
    return scores.sort_values(
        "risk_score", ascending=False, na_position="last").reset_index(drop=True)

In [307]:
# STAGE 2D — RIGHT RAIL
def participant_rail(user_id, phase="all", as_of=None):
    as_of = as_of or today_local()
    frame = daily_grid_metrics(phase)
    own = frame[frame["user_id"].eq(user_id)]
    scores = risk_scores(phase, as_of)
    score = scores[scores["user_id"].eq(user_id)]
    anchor = day1_dates().get(user_id)
    study_day_now = (as_of - anchor).days if anchor else None

    own_emas = ema_df[ema_df["user_id"].eq(user_id) & ema_df["status"].eq("completed")]
    last_ema = to_local(own_emas["sent_at"]).max() if not own_emas.empty else None
    age = last_sync_ages().get(user_id)
    covered = int(own["slots_covered"].sum())
    expected = int(own["slots_expected"].sum())
    wear_scored = own[own["gap_coverage"].notna()]

    return {
        "user_id": user_id,
        "phase": score["phase"].iloc[0] if len(score) else None,
        "study_day": study_day_now,
        "days_remaining": (STUDY_DAYS - 1 - study_day_now) if study_day_now is not None else None,
        "risk_score": score["risk_score"].iloc[0] if len(score) else None,
        "risk_components": score["risk_components"].iloc[0] if len(score) else None,
        "last_sync_age_h": age,
        "sync_measurable": sync_measurable(),
        "last_ema_at": last_ema,
        "slot_coverage_num": covered,
        "slot_coverage_den": expected,
        "slot_coverage_rate": suppress_rate(covered, expected, 1),
        "wear_days_scored": int(len(wear_scored)),
        "wear_days_met": int((wear_scored["gap_coverage"] >= BENCHMARKS["wear"]).sum()),
        "prompts_delivered": int(own["delivered_n"].sum()),
        "open_alerts": alerts_open_df[alerts_open_df["user_id"].eq(user_id)][
            ["severity", "rule_id", "fired_at", "payload"]].to_dict("records"),
    }

In [308]:
# STAGE 3 — LANES
def day_window(local_date):
    start = pd.Timestamp(local_date).tz_localize(PARTICIPANT_TZ)
    return start, start + pd.Timedelta(days=1)


def local_minutes(moment, local_date):
    if moment is None or pd.isna(moment):
        return None
    start, _ = day_window(local_date)
    return (pd.Timestamp(moment).tz_convert(PARTICIPANT_TZ) - start).total_seconds() / 60


def wear_lane(user_id, local_date):
    start, end = day_window(local_date)
    samples = heart_rate_df[
        heart_rate_df["user_id"].eq(user_id) & heart_rate_df["bpm"].gt(0)
    ] if not heart_rate_df.empty else heart_rate_df
    bins = np.zeros(MINUTES_PER_DAY, dtype=bool)
    if not samples.empty:
        stamps = pd.to_datetime(samples["timestamp"], utc=True).dt.tz_convert(PARTICIPANT_TZ)
        inside = samples[(stamps >= start) & (stamps < end)]
        if not inside.empty:
            minutes = ((pd.to_datetime(inside["timestamp"], utc=True).dt.tz_convert(PARTICIPANT_TZ) - start)
                       .dt.total_seconds() // 60).astype(int)
            bins[minutes.to_numpy()] = True

    gaps, run_start = [], None
    for minute in range(MINUTES_PER_DAY + 1) if bins.any() else []:
        filled = bins[minute] if minute < MINUTES_PER_DAY else True
        if filled:
            if run_start is not None and minute - run_start > WEAR_GAP_MIN:
                gaps.append({"start": run_start, "end": minute, "minutes": minute - run_start})
            run_start = None
        elif run_start is None:
            run_start = minute
    return {
        "user_id": user_id,
        "local_date": local_date,
        "has_data": bool(bins.any()),
        "covered_minutes": int(bins.sum()),
        "waking_window": [WAKING_WINDOW_START_HOUR * 60, WAKING_WINDOW_END_HOUR * 60],
        "waking_covered_minutes": int(bins[WAKING_WINDOW_START_HOUR * 60:WAKING_WINDOW_END_HOUR * 60].sum()),
        "gaps_gt_2h": gaps,
        "bins": bins,
    }


def sync_lane(user_id, local_date):
    start, end = day_window(local_date)
    own = wearable_sync_df[wearable_sync_df["user_id"].eq(user_id)] if not wearable_sync_df.empty else wearable_sync_df
    advances, carried_in = [], None
    if not own.empty:
        observed = pd.to_datetime(own["observed_at"], utc=True).dt.tz_convert(PARTICIPANT_TZ)
        before = own[observed < start]
        if not before.empty:
            carried_in = before.iloc[-1].to_dict()
        inside = own[(observed >= start) & (observed < end)]
        advances = [{
            "minute": local_minutes(row["observed_at"], local_date),
            "source": row["source"],
            "last_synced_at": row["last_synced_at"],
            "samples_written": row["samples_written"],
        } for row in inside.to_dict("records")]
    device = wearable_devices_df[wearable_devices_df["user_id"].eq(user_id)]
    return {
        "user_id": user_id,
        "local_date": local_date,
        "observed": bool(advances or carried_in),
        "measurable": sync_measurable(),
        "carried_in": carried_in,
        "advances": advances,
        "device_last_synced_at": device["last_synced_at"].iloc[0] if len(device) else None,
    }


def checkin_lane(user_id, local_date):
    start, end = day_window(local_date)
    own = ema_df[ema_df["user_id"].eq(user_id)]
    stamps = to_local(own["sent_at"])
    today = own[(stamps >= start) & (stamps < end)]
    scored = ema_completeness(today).set_index("ema_id") if not today.empty else None
    marks = []
    for row in today.to_dict("records"):
        marks.append({
            "ema_id": row["id"],
            "minute": local_minutes(row["sent_at"], local_date),
            "ema_type": row["ema_type"],
            "status": row["status"],
            "completeness": (scored.at[row["id"], "completeness"] if scored is not None else None),
            "missing_b1b2": bool(scored.at[row["id"], "missing_b1b2"]) if scored is not None else None,
        })

    own_reminders = checkin_reminders_df[checkin_reminders_df["user_id"].eq(user_id)]
    reminder_stamps = to_local(own_reminders["sent_at"])
    today_reminders = own_reminders[(reminder_stamps >= start) & (reminder_stamps < end)]
    ticks = [{
        "minute": local_minutes(row["sent_at"], local_date),
        "daily_count_at_send": row["daily_count_at_send"],
    } for row in today_reminders.to_dict("records")]

    slots = [
        {"index": index, "start": local_minutes(bounds[0], local_date), "end": local_minutes(bounds[1], local_date)}
        for index, bounds in enumerate(scheduled_slot_bounds(local_date))
    ]
    return {"user_id": user_id, "local_date": local_date, "slots": slots,
            "emas": marks, "reminders": ticks}

In [309]:
# STAGE 3 — DECISION, DELIVERY AND MSSD LANES
def classify_decision(row):
    reason = row["trigger_reason"]
    if reason in INELIGIBLE_REASONS:
        return reason
    if row["send_prompt"]:
        return "eligible-sent"
    if pd.notna(row["randomization_draw"]):
        return "eligible-not-sent"
    return "unclassified"


def participant_decisions(user_id, local_date=None):
    own = jitai_log_df[jitai_log_df["user_id"].eq(user_id)].copy()
    if own.empty:
        return own.assign(local_date=[], minute=[], outcome=[])
    anchor = own["decision_made_at"].fillna(own["triggered_at"])
    own["local_date"] = to_local(anchor).dt.date
    if local_date is not None:
        own = own[own["local_date"].eq(local_date)]
    own["minute"] = [local_minutes(m, d) for m, d in zip(anchor.loc[own.index], own["local_date"])]
    own["outcome"] = [classify_decision(row) for row in own.to_dict("records")]
    return own


def decision_lane(user_id, local_date):
    rows = participant_decisions(user_id, local_date)
    return [{
        "jitai_log_id": row["id"],
        "decision_point_id": row["decision_point_id"],
        "minute": row["minute"],
        "outcome": row["outcome"],
        "observed_mssd": row["observed_mssd"],
        "randomization_draw": row["randomization_draw"],
        "trigger_reason": row["trigger_reason"],
    } for row in rows.to_dict("records")]


def delivery_lane(user_id, local_date):
    rows = participant_decisions(user_id, local_date)
    sent = rows[rows["send_prompt"].astype(bool)] if not rows.empty else rows
    if sent.empty:
        return []
    events = engagement_log_df[engagement_log_df["jitai_log_id"].isin(set(sent["id"]))]
    by_log = {}
    for log_id, event_type in zip(events["jitai_log_id"], events["event_type"]):
        by_log.setdefault(log_id, set()).add(event_type)
    linked = ema_df[ema_df["source_jitai_log_id"].isin(set(sent["id"])) & ema_df["ema_type"].isin(OUTCOME_TYPES)]
    ema_by_log = {row["source_jitai_log_id"]: row for row in linked.to_dict("records")}

    out = []
    for row in sent.to_dict("records"):
        linked_ema = ema_by_log.get(row["id"])
        out.append({
            "jitai_log_id": row["id"],
            "push_sent_minute": local_minutes(row["push_sent_at"], local_date),
            "device_received_minute": local_minutes(row["device_received_at"], local_date),
            "receipt_reported_minute": local_minutes(row["receipt_reported_at"], local_date),
            "delivery_status": row["delivery_status"],
            "delivery_error": row["delivery_error"],
            "receipt_platform": row["receipt_platform"],
            "receipt_app_state": row["receipt_app_state"],
            "message_arm": row["message_arm"],
            "engagement": sorted(by_log.get(row["id"], set())),
            "outcome_window": [
                local_minutes(linked_ema["outcome_window_start"], local_date) if linked_ema else None,
                local_minutes(linked_ema["outcome_window_end"], local_date) if linked_ema else None,
            ],
            "linked_ema_id": linked_ema["id"] if linked_ema else None,
            "linked_ema_responded_minute": local_minutes(linked_ema["responded_at"], local_date) if linked_ema else None,
        })
    return out


def mssd_lane(user_id, local_date):
    rows = participant_decisions(user_id, local_date)
    out = []
    for row in rows.to_dict("records"):
        threshold = row["threshold_at_decision"]
        eligible = pd.notna(row["randomization_draw"])
        observed = row["observed_mssd"]
        out.append({
            "jitai_log_id": row["id"],
            "minute": row["minute"],
            "observed_mssd": observed,
            "threshold": threshold,
            "threshold_source": row["threshold_source"],
            "eligible": bool(eligible),
            "unexplained": bool(
                row["threshold_source"] == "engine"
                and pd.notna(observed) and pd.notna(threshold)
                and observed > threshold and not eligible
            ),
        })
    return out


def timeline(user_id, days=7, end=None):
    end = end or today_local()
    return [
        {
            "local_date": end - timedelta(days=offset),
            "wear": wear_lane(user_id, end - timedelta(days=offset)),
            "sync": sync_lane(user_id, end - timedelta(days=offset)),
            "checkins": checkin_lane(user_id, end - timedelta(days=offset)),
            "decisions": decision_lane(user_id, end - timedelta(days=offset)),
            "delivery": delivery_lane(user_id, end - timedelta(days=offset)),
            "mssd": mssd_lane(user_id, end - timedelta(days=offset)),
        }
        for offset in reversed(range(days))
    ]

In [310]:
# STAGE 3B — DELIVERY FUNNEL
def delivery_funnel(user_id):
    own = jitai_log_df[jitai_log_df["user_id"].eq(user_id)]
    sent = own[own["send_prompt"].astype(bool)]
    pushed = sent[sent["push_sent_at"].notna()]
    received = pushed[pushed["device_received_at"].notna()]
    reported = received[received["receipt_reported_at"].notna()]
    engaged_ids = set(engagement_log_df.loc[
        engagement_log_df["jitai_log_id"].isin(set(reported["id"])), "jitai_log_id"])
    stages = [
        ("send_prompt", len(sent)),
        ("push_sent_at", len(pushed)),
        ("device_received_at", len(received)),
        ("receipt_reported_at", len(reported)),
        ("engagement_event", len(engaged_ids)),
    ]
    rows = []
    for index, (name, count) in enumerate(stages):
        previous = stages[index - 1][1] if index else None
        rows.append({
            "stage": name,
            "n": count,
            "drop_from_previous": (previous - count) if previous is not None else None,
            "drop_pct": ((previous - count) / previous) if previous else None,
        })
    splits = {}
    for column in ("receipt_platform", "receipt_app_state"):
        splits[column] = (
            received.groupby(received[column].fillna("unknown")).size().to_dict()
            if not received.empty else {}
        )
    errors = (
        sent.loc[sent["delivery_error"].notna() & sent["delivery_error"].ne(""), "delivery_error"]
        .value_counts().to_dict()
    )
    return {
        "user_id": user_id,
        "stages": pd.DataFrame(rows),
        "splits": splits,
        "delivery_errors": errors,
    }

In [311]:
# STAGE 3C — ITEM COMPLETENESS MATRIX
def completeness_matrix(user_id, local_date):
    start, end = day_window(local_date)
    own = ema_df[ema_df["user_id"].eq(user_id)]
    stamps = to_local(own["sent_at"])
    today = own[(stamps >= start) & (stamps < end)]
    if today.empty:
        return pd.DataFrame()
    scored = ema_completeness(today)
    maps = answered_maps(today["id"])
    columns = sorted({sub_id for askable in scored["askable"] for sub_id in askable})
    rows = []
    for record in scored.to_dict("records"):
        answers = maps.get(record["ema_id"], {})
        askable = set(record["askable"])
        cells = {
            sub_id: ("answered" if sub_id in answers else "not answered")
            if sub_id in askable else "not applicable"
            for sub_id in columns
        }
        rows.append({
            "ema_id": record["ema_id"],
            "ema_type": record["ema_type"],
            "completeness": record["completeness"],
            "missing_b1b2": record["missing_b1b2"],
            "item_bank_version": record["item_bank_version"],
            **cells,
        })
    return pd.DataFrame(rows)

In [312]:
# 4 — ALERT RULES
ALERT_SPEC_SEVERITY = {
    "runin_prompt_sent": "high", "randomization_mismatch": "high",
    "cooldown_violation": "high", "cap_exceeded": "high", "scheduler_silent": "high",
    "sync_stale": "medium", "no_checkin_72h": "medium", "wear_low": "medium",
    "mssd_signal_missing": "medium", "threshold_never_fires": "medium",
    "clock_skew": "low", "enrollment_flip": "info",
}
NO_CHECKIN_HOURS = 72
SCHEDULER_SILENT_FLOOR = 3
WEAR_LOW_CONSECUTIVE_DAYS = 3
MSSD_MISSING_FRACTION = 0.20
THRESHOLD_NEVER_FIRES_MIN_POINTS = 20
CLOCK_SKEW_MINUTES = 5


def finding(rule_id, user_id=None, local_date=None, stage=None, **payload):
    return {
        "rule_id": rule_id,
        "severity": ALERT_SPEC_SEVERITY[rule_id],
        "stage": stage,
        "user_id": user_id,
        "local_date": local_date,
        "payload": payload,
    }


def evaluate_rules(phase="all", as_of=None):
    as_of = as_of or today_local()
    frame = daily_grid_metrics(phase)
    logs = decisions(phase, mrt_only=False)
    out = []

    if not logs.empty:
        run_in_keys = {(u, d) for u, d, r in zip(frame["user_id"], frame["local_date"], frame["is_run_in"]) if r}
        for row in logs[logs["send_prompt"].astype(bool)].to_dict("records"):
            if (row["user_id"], row["local_date"]) in run_in_keys:
                out.append(finding("runin_prompt_sent", row["user_id"], row["local_date"], 1,
                                   jitai_log_id=row["id"]))
        drawn = logs[logs["randomization_draw"].notna()]
        for row in drawn.to_dict("records"):
            if bool(row["send_prompt"]) != (row["randomization_draw"] < row["randomization_probability"]):
                out.append(finding("randomization_mismatch", row["user_id"], row["local_date"], 1,
                                   draw=row["randomization_draw"], p=row["randomization_probability"],
                                   send_prompt=bool(row["send_prompt"])))
        _, pairs = cooldown_violations(logs)
        for pair in pairs:
            out.append(finding("cooldown_violation", pair["user_id"], None, 1,
                               gap_min=pair["gap_min"], threshold_min=JITAI_COOLDOWN_MINUTES,
                               earlier_id=pair["earlier_id"], later_id=pair["later_id"]))

    for row in frame.to_dict("records"):
        if row["delivered_n"] > DAILY_PROMPT_CAP:
            out.append(finding("cap_exceeded", row["user_id"], row["local_date"], 1,
                               delivered_n=row["delivered_n"], cap=DAILY_PROMPT_CAP))
        if row["slots_silent"] >= SCHEDULER_SILENT_FLOOR:
            out.append(finding("scheduler_silent", row["user_id"], row["local_date"], 2,
                               slots_silent=int(row["slots_silent"])))

    measurable = sync_measurable()
    if not measurable:
        out.append(finding("sync_stale", None, None, 2, measurable=False,
                           detail="nothing writes the sync clock; cohort-scoped, not per participant"))
    else:
        for uid, age in last_sync_ages().items():
            if age > RISK_SYNC_STALE_HOURS:
                out.append(finding("sync_stale", uid, None, 2, last_sync_age_h=round(age, 1)))

    slots = scheduled_slots(set(frame["user_id"]))
    last_ema = slots.groupby("user_id")["local_date"].max().to_dict() if not slots.empty else {}
    for uid in sorted(set(frame["user_id"])):
        latest = last_ema.get(uid)
        hours = ((as_of - latest).days * 24) if latest else None
        if hours is None or hours > NO_CHECKIN_HOURS:
            out.append(finding("no_checkin_72h", uid, None, 2,
                               last_scheduled_ema=latest, hours=hours))

        own = frame[frame["user_id"].eq(uid)].sort_values("local_date")
        streak = 0
        for coverage in own["gap_coverage"]:
            streak = streak + 1 if pd.notna(coverage) and coverage < RISK_WEAR_FLOOR else 0
            if streak >= WEAR_LOW_CONSECUTIVE_DAYS:
                out.append(finding("wear_low", uid, None, 2, consecutive_days=streak))
                break

        week = own[own["local_date"].gt(as_of - timedelta(days=RISK_TRAILING_DAYS))]
        scheduled_n = int(week["slots_covered"].sum())
        missing = int(week["ema_missing_b1b2_n"].sum())
        if scheduled_n and missing / scheduled_n > MSSD_MISSING_FRACTION:
            out.append(finding("mssd_signal_missing", uid, None, 3,
                               missing=missing, emas=scheduled_n))

        if not logs.empty:
            recent = logs[logs["user_id"].eq(uid) & logs["local_date"].gt(as_of - timedelta(days=RISK_TRAILING_DAYS))]
            if len(recent) >= THRESHOLD_NEVER_FIRES_MIN_POINTS and recent["randomization_draw"].notna().sum() == 0:
                out.append(finding("threshold_never_fires", uid, None, 3,
                                   decision_points=int(len(recent)),
                                   caveat="also fires under an unrecorded distress suspension"))

    telemetry = phone_telemetry_df[phone_telemetry_df["user_id"].isin(set(frame["user_id"]))]
    if not telemetry.empty:
        skew = (pd.to_datetime(telemetry["recorded_at"], utc=True)
                - pd.to_datetime(telemetry["occurred_at"], utc=True)).dt.total_seconds() / 60
        for uid, group in skew.groupby(telemetry["user_id"]):
            p95 = float(np.percentile(group.dropna(), 95)) if group.notna().any() else None
            if p95 is not None and (p95 > CLOCK_SKEW_MINUTES or p95 < 0):
                out.append(finding("clock_skew", uid, None, 3, p95_minutes=round(p95, 2)))

    flipped = users_df[users_df["enrolled_at"].notna() & ~users_df["is_enrolled"].astype(bool)]
    for uid in flipped["user_id"]:
        if uid in set(frame["user_id"]) or uid in cohort_users(phase):
            out.append(finding("enrollment_flip", uid, None, 1))

    order = {"high": 0, "medium": 1, "low": 2, "info": 3}
    result = pd.DataFrame(out, columns=["rule_id", "severity", "stage", "user_id", "local_date", "payload"])
    if result.empty:
        return result
    return result.assign(rank=result["severity"].map(order)).sort_values(
        ["rank", "rule_id"]).drop(columns="rank").reset_index(drop=True)

In [313]:
risk_scores("all")

,user_id,phase,study_day_now,risk_score,risk_components,sync_measurable
0,1003,mrt,18,31.0,"{'ema_stale': 15, 'low_coverage': 14, 'sync_st...",True
1,1009,mrt,24,15.0,"{'ema_stale': 0, 'low_coverage': 4, 'sync_stal...",True
2,1005,run_in,6,11.0,"{'ema_stale': 3, 'low_coverage': 8, 'sync_stal...",True
3,1010,mrt,28,11.0,"{'ema_stale': 0, 'low_coverage': 4, 'sync_stal...",True
4,1007,mrt,15,10.0,"{'ema_stale': 0, 'low_coverage': 6, 'sync_stal...",True
5,1001,mrt,34,6.0,"{'ema_stale': 0, 'low_coverage': 6, 'sync_stal...",True
6,1008,mrt,22,6.0,"{'ema_stale': 0, 'low_coverage': 2, 'sync_stal...",True
7,1006,run_in,5,4.0,"{'ema_stale': 0, 'low_coverage': 4, 'sync_stal...",True
8,1012,mrt,12,4.0,"{'ema_stale': 0, 'low_coverage': 4, 'sync_stal...",True
9,1013,mrt,16,4.0,"{'ema_stale': 0, 'low_coverage': 4, 'sync_stal...",True


In [314]:
grid_frame("slot_coverage", "all")

,0,1,2,3,4,5,6,7,8,9,...,25,26,27,28,29,30,31,32,33,34
1001,0.500000,0.333333,0.333333,0.333333,0.666667,0.500000,0.666667,0.666667,0.666667,0.666667,...,0.333333,0.666667,0.666667,0.833333,0.333333,0.666667,0.5,0.5,0.333333,0.333333
1002,0.500000,0.000000,0.500000,0.333333,0.666667,0.333333,0.666667,1.000000,0.500000,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1003,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1009,0.333333,0.666667,0.833333,0.333333,1.000000,0.500000,0.333333,0.666667,0.333333,0.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1005,0.500000,0.500000,0.333333,0.166667,0.333333,0.500000,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1010,0.500000,0.666667,0.333333,0.333333,0.666667,0.500000,0.833333,0.000000,0.500000,0.333333,...,0.833333,0.333333,0.666667,0.500000,NaN,NaN,NaN,NaN,NaN,NaN
1007,0.333333,0.500000,0.500000,0.500000,0.666667,0.333333,0.666667,0.833333,0.666667,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1008,0.833333,0.666667,1.000000,0.833333,0.666667,1.000000,0.333333,0.500000,0.333333,0.333333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1006,0.500000,0.500000,0.500000,0.333333,0.500000,0.333333,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1012,0.500000,0.166667,0.000000,0.333333,0.333333,0.333333,0.500000,0.666667,0.000000,0.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [315]:
slot_split("all")

,user_id,covered,reminded,silent,days,covered_pct,reminded_pct,silent_pct
0,1001,21,19,2,7,0.500000,0.452381,0.047619
1,1003,0,0,42,7,0.000000,0.000000,1.000000
2,1004,24,17,1,7,0.571429,0.404762,0.023810
3,1005,14,27,1,7,0.333333,0.642857,0.023810
4,1006,16,18,2,6,0.444444,0.500000,0.055556
5,1007,21,20,1,7,0.500000,0.476190,0.023810
6,1008,24,16,2,7,0.571429,0.380952,0.047619
7,1009,25,14,3,7,0.595238,0.333333,0.071429
8,1010,22,19,1,7,0.523810,0.452381,0.023810
9,1011,26,14,2,7,0.619048,0.333333,0.047619


In [316]:
GRID_USER = int(risk_scores("all")["user_id"].iloc[0])
participant_rail(GRID_USER)

{'user_id': 1003,
 'phase': 'mrt',
 'study_day': 18,
 'days_remaining': 16,
 'risk_score': np.float64(31.0),
 'risk_components': {'ema_stale': 15,
  'low_coverage': 14,
  'sync_stale': 2,
  'low_wear': 0,
  'missing_signal': 0,
  'open_critical': 0},
 'last_sync_age_h': 29.778335185555555,
 'sync_measurable': True,
 'last_ema_at': None,
 'slot_coverage_num': 0,
 'slot_coverage_den': 114,
 'slot_coverage_rate': None,
 'wear_days_scored': 14,
 'wear_days_met': 14,
 'prompts_delivered': 0,
 'open_alerts': []}

In [317]:
strips = timeline(GRID_USER, days=7)
pd.DataFrame([
    {
        "local_date": day["local_date"],
        "wear_minutes": day["wear"]["covered_minutes"],
        "gaps_gt_2h": len(day["wear"]["gaps_gt_2h"]),
        "sync_observed": day["sync"]["observed"],
        "emas": len(day["checkins"]["emas"]),
        "reminders": len(day["checkins"]["reminders"]),
        "decisions": len(day["decisions"]),
        "delivered": len(day["delivery"]),
        "unexplained_mssd": sum(1 for point in day["mssd"] if point["unexplained"]),
    }
    for day in strips
])

,local_date,wear_minutes,gaps_gt_2h,sync_observed,emas,reminders,decisions,delivered,unexplained_mssd
0,2026-09-10,793,1,True,0,0,0,0,0
1,2026-09-11,793,1,True,0,0,0,0,0
2,2026-09-12,803,1,True,0,0,0,0,0
3,2026-09-13,769,1,True,0,0,0,0,0
4,2026-09-14,751,1,True,0,0,0,0,0
5,2026-09-15,767,1,True,0,0,0,0,0
6,2026-09-16,775,1,True,0,0,0,0,0


In [318]:
funnel = delivery_funnel(GRID_USER)
print("splits:", funnel["splits"])
print("delivery errors:", funnel["delivery_errors"])
funnel["stages"]

splits: {'receipt_platform': {}, 'receipt_app_state': {}}
delivery errors: {}


,stage,n,drop_from_previous,drop_pct
0,send_prompt,0,NaN,None
1,push_sent_at,0,0.0,None
2,device_received_at,0,0.0,None
3,receipt_reported_at,0,0.0,None
4,engagement_event,0,0.0,None


In [319]:
completeness_matrix(GRID_USER, max(
    (day["local_date"] for day in strips if day["checkins"]["emas"]), default=today_local()))

""


In [320]:
findings = evaluate_rules("all")
print("findings:", len(findings))
findings.groupby(["severity", "rule_id"]).size()

findings: 53


severity  rule_id            
high      cap_exceeded            2
          cooldown_violation      9
          runin_prompt_sent      12
          scheduler_silent       21
info      enrollment_flip         1
low       clock_skew              2
medium    mssd_signal_missing     1
          no_checkin_72h          2
          sync_stale              2
          wear_low                1
dtype: int64

In [321]:
findings

,rule_id,severity,stage,user_id,local_date,payload
0,cap_exceeded,high,1,1008,2026-08-30,"{'delivered_n': 8, 'cap': 4}"
1,cap_exceeded,high,1,1008,2026-09-12,"{'delivered_n': 6, 'cap': 4}"
2,cooldown_violation,high,1,1007,None,"{'gap_min': 30.1, 'threshold_min': 60, 'earlie..."
3,cooldown_violation,high,1,1008,None,"{'gap_min': 15.1, 'threshold_min': 60, 'earlie..."
4,cooldown_violation,high,1,1008,None,"{'gap_min': 2.9, 'threshold_min': 60, 'earlier..."
5,cooldown_violation,high,1,1008,None,"{'gap_min': 0.2, 'threshold_min': 60, 'earlier..."
6,cooldown_violation,high,1,1008,None,"{'gap_min': 18.0, 'threshold_min': 60, 'earlie..."
7,cooldown_violation,high,1,1008,None,"{'gap_min': 3.0, 'threshold_min': 60, 'earlier..."
8,cooldown_violation,high,1,1008,None,"{'gap_min': 0.1, 'threshold_min': 60, 'earlier..."
9,cooldown_violation,high,1,1008,None,"{'gap_min': 35.8, 'threshold_min': 60, 'earlie..."


In [322]:
# STAGE 1 PLOTS — PALETTE
import json

import plotly.graph_objects as go
from plotly.subplots import make_subplots

VIZ_MODE = "light"

PALETTE = {
    "light": {
        "surface": "#fcfcfb", "text": "#0b0b0b", "text_secondary": "#52514e",
        "muted": "#898781", "grid": "#e1e0d9", "axis": "#c3c2b7",
        "series_1": "#2a78d6", "series_2": "#eb6834",
        "ordinal": ["#86b6ef", "#5598e7", "#2a78d6", "#184f95"],
        "band": "#f0efec",
    },
    "dark": {
        "surface": "#1a1a19", "text": "#ffffff", "text_secondary": "#c3c2b7",
        "muted": "#898781", "grid": "#2c2c2a", "axis": "#383835",
        "series_1": "#3987e5", "series_2": "#d95926",
        "ordinal": ["#cde2fb", "#86b6ef", "#3987e5", "#184f95"],
        "band": "#383835",
    },
}
STATUS = {"good": "#0ca30c", "warning": "#fab219", "serious": "#ec835a", "critical": "#d03b3b"}
STATUS_ICON = {"good": "\u25cf", "warning": "\u25b2", "serious": "\u25b2", "critical": "\u25a0"}
KS_MIN_DRAWS = 20
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'


def ink(role):
    return PALETTE[VIZ_MODE][role]


def base_layout(fig, height, title=None, margin=None):
    fig.update_layout(
        template="none",
        paper_bgcolor=ink("surface"),
        plot_bgcolor=ink("surface"),
        font=dict(family=FONT, size=12, color=ink("text_secondary")),
        title=dict(text=title, font=dict(size=14, color=ink("text")), x=0, xanchor="left") if title else None,
        height=height,
        margin=margin or dict(l=140, r=40, t=44 if title else 12, b=28),
        showlegend=False,
        hoverlabel=dict(bgcolor=ink("surface"), font=dict(family=FONT, color=ink("text"))),
    )
    fig.update_xaxes(showgrid=False, zeroline=False, linecolor=ink("axis"),
                     tickfont=dict(color=ink("muted")))
    fig.update_yaxes(showgrid=False, zeroline=False, linecolor=ink("axis"),
                     tickfont=dict(color=ink("muted")))
    return fig


def pct(value, digits=0):
    return "-" if value is None or pd.isna(value) else f"{value * 100:.{digits}f}%"

In [323]:
# STAGE 1 PLOTS — PLATFORM SERIES FOR THE PROMPT-RESPONSE SPARKLINE
def platform_series_14d(phase="all", as_of=None):
    as_of = as_of or today_local()
    window_start = as_of - timedelta(days=SERIES_DAYS - 1)
    rows_jitai = decisions(phase, mrt_only=False)
    delivered = (
        rows_jitai[rows_jitai["send_prompt"].astype(bool) & rows_jitai["device_received_at"].notna()]
        if not rows_jitai.empty else rows_jitai
    )
    responded = responded_logs(delivered) if not delivered.empty else set()
    out = []
    for offset in range(SERIES_DAYS):
        local = window_start + timedelta(days=offset)
        day = delivered[delivered["local_date"].eq(local)] if not delivered.empty else delivered
        row = {"date": local}
        for platform in ("ios", "android"):
            group = day[day["receipt_platform"].fillna("unknown").eq(platform)] if len(day) else day
            n = int(len(group))
            k = int(group["id"].isin(responded).sum()) if n else 0
            row[f"{platform}_num"] = k
            row[f"{platform}_den"] = n
            row[platform] = suppress_rate(k, n, int(group["user_id"].nunique()) if n else 0)
        out.append(row)
    return pd.DataFrame(out)

In [324]:
# STAGE 1A — BENCHMARK CARD (BULLET + WILSON CI + 14-DAY SPARKLINE)
SPARK_COLUMN = {
    "slot_coverage": ("slot_coverage", "slots_covered", "slots_expected"),
    "prompt_response": ("prompt_response", "responded_n", "delivered_n"),
    "wear": ("wear_pass_rate", "wear_days_met", "wear_days_scored"),
}


def benchmark_state(entry):
    if not entry["measurable"]:
        return "unmeasurable"
    if entry["suppressed"]:
        return "suppressed"
    return "good" if entry["value"] >= entry["target"] else "critical"


def plot_benchmark_card(name, entry, series=None, platform=None, title=None):
    state = benchmark_state(entry)
    label = title or entry.get("label", name.replace("_", " ").title())
    has_spark = series is not None and not series.empty
    fig = make_subplots(
        rows=2 if has_spark else 1, cols=1, row_heights=[0.62, 0.38] if has_spark else [1.0],
        vertical_spacing=0.28, shared_xaxes=False,
    )

    if state in ("good", "critical"):
        fig.add_trace(go.Bar(
            x=[entry["value"]], y=[label], orientation="h", width=0.34,
            marker=dict(color=ink("series_1"), line=dict(width=0)),
            error_x=dict(
                type="data", symmetric=False,
                array=[entry["wilson_high"] - entry["value"]],
                arrayminus=[entry["value"] - entry["wilson_low"]],
                color=ink("text_secondary"), thickness=2, width=6),
            hovertemplate=(
                f"<b>{label}</b><br>value %{{x:.1%}}<br>"
                f"95% CI {pct(entry['wilson_low'], 1)}-{pct(entry['wilson_high'], 1)}<br>"
                f"{entry['numerator']}/{entry['denominator']} · {entry['participants']} participants"
                "<extra></extra>"),
            showlegend=False), row=1, col=1)
        headline = pct(entry["value"], 1)
        detail = f"{entry['numerator']}/{entry['denominator']}"
    else:
        numerator = entry["numerator"] if entry["numerator"] is not None else "-"
        denominator = entry["denominator"] if entry["denominator"] is not None else "-"
        fig.add_trace(go.Bar(
            x=[0], y=[label], orientation="h", width=0.34,
            marker=dict(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=False), row=1, col=1)
        headline = f"{numerator}/{denominator}"
        detail = "no source" if state == "unmeasurable" else "rate withheld: thin denominator"

    if entry["target"] is not None:
        fig.add_shape(
            type="line", x0=entry["target"], x1=entry["target"], y0=-0.45, y1=0.45,
            line=dict(color=ink("text"), width=2, dash="dot"), row=1, col=1)
        fig.add_annotation(
            x=entry["target"], y=0.55, yref="y", text=f"target {pct(entry['target'])}",
            showarrow=False, font=dict(size=10, color=ink("muted")), row=1, col=1)

    badge = {"good": "good", "critical": "critical",
             "suppressed": "warning", "unmeasurable": "warning"}[state]
    fig.add_annotation(
        x=1.0, y=1.0, xref="paper", yref="paper", xanchor="right", yanchor="bottom",
        yshift=26, showarrow=False,
        text=(f"<b style='font-size:20px'>{headline}</b>  "
              f"<span style='color:{STATUS[badge]}'>{STATUS_ICON[badge]}</span> "
              f"<span style='color:{ink('muted')}'>{detail}</span>"),
        font=dict(family=FONT, size=12, color=ink("text")))

    if has_spark:
        traces = platform if platform else [(None, ink("series_1"), "")]
        for column, color, trace_label in traces:
            column = column or "value"
            fig.add_trace(go.Scatter(
                x=series["date"], y=series[column], mode="lines",
                line=dict(color=color, width=2, shape="spline", smoothing=0.6),
                connectgaps=False, name=trace_label or label,
                hovertemplate=f"{trace_label or label}<br>%{{x|%b %d}}: %{{y:.0%}}<extra></extra>",
                showlegend=bool(platform)), row=2, col=1)
            tail = series[series[column].notna()]
            if len(tail) and trace_label:
                fig.add_annotation(
                    x=tail["date"].iloc[-1], y=tail[column].iloc[-1], text=f" {trace_label}",
                    xanchor="left", showarrow=False, font=dict(size=10, color=ink("muted")),
                    row=2, col=1)
        drawn = [column or "value" for column, _, _ in traces]
        if series[drawn].isna().all().all():
            fig.add_annotation(
                x=0.5, y=0.5, xref="x domain", yref="y domain", text="14-day rate suppressed",
                showarrow=False, font=dict(size=10, color=ink("muted")), row=2, col=1)
        fig.update_yaxes(range=[0, 1], showticklabels=False, row=2, col=1)
        fig.update_xaxes(showticklabels=True, tickformat="%b %d", nticks=4,
                         linecolor=ink("grid"), row=2, col=1)

    fig.update_xaxes(range=[0, 1], tickformat=".0%", row=1, col=1)
    fig.update_yaxes(showticklabels=True, tickfont=dict(color=ink("text"), size=12), row=1, col=1)
    base_layout(fig, 210 if has_spark else 140,
                margin=dict(l=150, r=40, t=64, b=34 if has_spark else 28))
    fig.update_layout(
        bargap=0.5, showlegend=bool(platform),
        legend=dict(orientation="h", y=-0.34, x=0, font=dict(size=10, color=ink("text_secondary"))))
    return fig


def plot_benchmark_cards(board):
    figures = {}
    series = board["series_14d"]
    platforms = platform_series_14d(board["phase"])
    for name, entry in board["benchmarks"].items():
        spark = None
        platform = None
        if name in SPARK_COLUMN:
            column, num, den = SPARK_COLUMN[name]
            spark = series[["date", column, num, den]].rename(columns={column: "value"})
        if name == "prompt_response":
            spark = platforms.rename(columns={"ios": "ios_rate", "android": "android_rate"})
            platform = [("ios_rate", ink("series_1"), "iOS"),
                        ("android_rate", ink("series_2"), "Android")]
        figures[name] = plot_benchmark_card(name, entry, spark, platform)
    return figures


def benchmark_card_table(board):
    return benchmark_table(board)

In [325]:
# STAGE 1B — ENROLLMENT FUNNEL
def plot_funnel(board):
    rows = board["funnel"].copy()
    stages = rows[rows["stage"].ne("withdrew")]
    withdrew = rows[rows["stage"].eq("withdrew")]
    ramp = ink("ordinal")

    fig = go.Figure()
    for index, row in enumerate(stages.to_dict("records")):
        measurable = bool(row["measurable"])
        value = row["n"] if measurable else 0
        fig.add_trace(go.Bar(
            x=[value], y=[row["stage"]], orientation="h", width=0.62,
            marker=dict(
                color=ramp[min(index, len(ramp) - 1)] if measurable else "rgba(0,0,0,0)",
                line=dict(color=ink("axis") if not measurable else ink("surface"), width=2)),
            hovertemplate=(f"<b>{row['stage']}</b><br>n = {row['n']}<extra></extra>" if measurable
                           else f"<b>{row['stage']}</b><br>not measurable<extra></extra>"),
            showlegend=False))
        fig.add_annotation(
            x=value, y=row["stage"], xanchor="left", xshift=8, showarrow=False,
            text=(f"<b>{int(row['n'])}</b>" if measurable else
                  f"<span style='color:{STATUS['warning']}'>{STATUS_ICON['warning']}</span> not recorded"),
            font=dict(size=12, color=ink("text")))

    if len(withdrew):
        row = withdrew.iloc[0]
        fig.add_trace(go.Bar(
            x=[row["n"]], y=["withdrew"], orientation="h", width=0.42,
            marker=dict(color=STATUS["critical"], line=dict(color=ink("surface"), width=2)),
            hovertemplate=f"<b>withdrew</b><br>n = {row['n']}<extra></extra>", showlegend=False))
        fig.add_annotation(
            x=row["n"], y="withdrew", xanchor="left", xshift=8, showarrow=False,
            text=(f"<span style='color:{STATUS['critical']}'>{STATUS_ICON['critical']}</span> "
                  f"<b>{int(row['n'])}</b> dropout"),
            font=dict(size=12, color=ink("text")))

    order = list(stages["stage"])[::-1]
    if len(withdrew):
        order = ["withdrew"] + order
    fig.update_yaxes(categoryorder="array", categoryarray=order,
                     tickfont=dict(color=ink("text"), size=12))
    fig.update_xaxes(showgrid=True, gridcolor=ink("grid"), rangemode="tozero")
    base_layout(fig, 250, f"Enrollment funnel - {board['phase']} (n={board['n_participants']})",
                margin=dict(l=150, r=120, t=50, b=30))
    fig.update_layout(bargap=0.35)
    return fig

In [326]:
# STAGE 1C — MRT INTEGRITY STRIP
def plot_integrity_bullet(entry, label, band=None, target_point=None, width_range=(0, 1)):
    fig = go.Figure()
    if band:
        fig.add_shape(type="rect", x0=band[0], x1=band[1], y0=-0.5, y1=0.5,
                      fillcolor=ink("band"), line=dict(width=0), layer="below")
        fig.add_annotation(x=(band[0] + band[1]) / 2, y=0.62,
                           text=f"target band {pct(band[0])}-{pct(band[1])}",
                           showarrow=False, font=dict(size=10, color=ink("muted")))
    value = entry.get("value")
    alarm = bool(entry.get("alarm"))
    if entry.get("contradiction") or alarm:
        badge = "critical"
    elif value is None or entry.get("wilson_low") is None:
        badge = "warning"
    else:
        badge = "good"
    if value is not None:
        fig.add_trace(go.Bar(
            x=[value], y=[label], orientation="h", width=0.3,
            marker=dict(color=ink("series_1"), line=dict(width=0)),
            error_x=(dict(type="data", symmetric=False,
                          array=[entry["wilson_high"] - value],
                          arrayminus=[value - entry["wilson_low"]],
                          color=ink("text_secondary"), thickness=2, width=6)
                     if entry.get("wilson_low") is not None else None),
            hovertemplate=(f"<b>{label}</b><br>%{{x:.1%}}<br>"
                           f"{entry.get('numerator')}/{entry.get('denominator')}<extra></extra>"),
            showlegend=False))
    if target_point is not None:
        fig.add_shape(type="line", x0=target_point, x1=target_point, y0=-0.42, y1=0.42,
                      line=dict(color=ink("text"), width=2, dash="dot"))
        fig.add_annotation(x=target_point, y=0.62, text=f"p = {target_point:g}",
                           showarrow=False, font=dict(size=10, color=ink("muted")))
    note = ""
    if entry.get("contradiction"):
        note = (f"  <span style='color:{STATUS['critical']}'>{STATUS_ICON['critical']}</span> "
                f"k &gt; n ({entry['numerator']}/{entry['denominator']})")
    fig.add_annotation(
        x=1.0, y=1.0, xref="paper", yref="paper", xanchor="right", yanchor="bottom",
        yshift=22, showarrow=False,
        text=(f"<b style='font-size:16px'>{pct(value, 1)}</b>  "
              f"<span style='color:{STATUS[badge]}'>{STATUS_ICON[badge]}</span> "
              f"<span style='color:{ink('muted')}'>"
              f"{entry.get('numerator')}/{entry.get('denominator')}</span>{note}"),
        font=dict(family=FONT, size=12, color=ink("text")))
    fig.update_xaxes(range=list(width_range), tickformat=".0%")
    fig.update_yaxes(tickfont=dict(color=ink("text"), size=12))
    base_layout(fig, 136, margin=dict(l=150, r=40, t=58, b=30))
    fig.update_layout(bargap=0.5)
    return fig


def plot_randomization_ecdf(integrity, draws):
    audit = integrity["randomization_audit"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1], mode="lines", name="Uniform(0,1)",
        line=dict(color=ink("muted"), width=2, dash="dot"),
        hovertemplate="Uniform(0,1)<br>%{x:.2f}<extra></extra>"))
    values = sorted(draws)
    if values:
        steps = [(index + 1) / len(values) for index in range(len(values))]
        fig.add_trace(go.Scatter(
            x=values, y=steps, mode="lines+markers", name="Observed draws",
            line=dict(color=ink("series_1"), width=2, shape="hv"),
            marker=dict(size=8, color=ink("series_1"),
                        line=dict(color=ink("surface"), width=2)),
            hovertemplate="Observed<br>draw %{x:.3f}<br>ECDF %{y:.2f}<extra></extra>"))
    else:
        fig.add_annotation(x=0.5, y=0.5, text="no draws recorded", showarrow=False,
                           font=dict(size=11, color=ink("muted")))
    alarm = audit["ks_p_value"] is not None and audit["ks_p_value"] < audit["alarm_p"]
    thin = audit["draws"] < KS_MIN_DRAWS
    badge = "critical" if (alarm or audit["mismatches"]) else ("warning" if thin else "good")
    if audit["ks_statistic"] is None:
        summary = "KS: no draws recorded"
    elif thin:
        summary = (f"KS D = {audit['ks_statistic']:.3f}, p = {audit['ks_p_value']:.3f} - "
                   f"{audit['draws']} draw(s), too few to test")
    else:
        summary = f"KS D = {audit['ks_statistic']:.3f}, p = {audit['ks_p_value']:.3f}"
    if audit["mismatches"]:
        summary += f" · {audit['mismatches']} mismatch(es)"
    fig.add_annotation(
        x=1.0, y=1.0, xref="paper", yref="paper", xanchor="right", yanchor="bottom",
        yshift=14, showarrow=False,
        text=(f"<span style='color:{STATUS[badge]}'>{STATUS_ICON[badge]}</span> {summary}"),
        font=dict(family=FONT, size=12, color=ink("text")))
    fig.update_xaxes(range=[0, 1], title=dict(text="randomization_draw",
                                              font=dict(size=11, color=ink("muted"))))
    fig.update_yaxes(range=[0, 1], showgrid=True, gridcolor=ink("grid"),
                     title=dict(text="cumulative share", font=dict(size=11, color=ink("muted"))))
    base_layout(fig, 300, "Randomization uniformity", margin=dict(l=70, r=40, t=54, b=50))
    fig.update_layout(showlegend=True,
                      legend=dict(orientation="h", y=-0.28, x=0,
                                  font=dict(size=11, color=ink("text_secondary"))))
    return fig


def plot_violation_counters(integrity, runin_sent):
    cooldown = integrity["cooldown"]
    eligibility = integrity["eligibility_rate"]
    tiles = [
        ("Cooldown violations", cooldown["violations"], cooldown["alarm"],
         f"consecutive sends &lt; {cooldown['threshold_min']} min"),
        ("Run-in prompts sent", runin_sent, runin_sent > 0,
         "protocol: no prompts before day 7"),
        ("Reason disagreement", eligibility["reason_eligible_but_no_draw"]
         + eligibility["reason_ineligible_with_draw"],
         eligibility["alarm_reason_disagreement"], "trigger_reason vs randomization_draw"),
    ]
    fig = make_subplots(rows=1, cols=len(tiles), horizontal_spacing=0.06)
    for index, (label, value, alarm, caption) in enumerate(tiles, start=1):
        suffix = "" if index == 1 else str(index)
        xref, yref = f"x{suffix} domain", f"y{suffix} domain"
        badge = "critical" if alarm else "good"
        fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers",
                                 marker=dict(size=0.1, color="rgba(0,0,0,0)"),
                                 hoverinfo="skip", showlegend=False), row=1, col=index)
        fig.add_annotation(
            x=0.5, y=0.88, xref=xref, yref=yref, showarrow=False, yanchor="top",
            text=f"<b style='font-size:30px;color:{STATUS[badge]}'>{int(value)}</b>",
            font=dict(family=FONT))
        fig.add_annotation(
            x=0.5, y=0.30, xref=xref, yref=yref, showarrow=False,
            text=(f"<span style='color:{STATUS[badge]}'>{STATUS_ICON[badge]}</span> "
                  f"<b>{label}</b>"),
            font=dict(family=FONT, size=12, color=ink("text")))
        fig.add_annotation(
            x=0.5, y=0.02, xref=xref, yref=yref, showarrow=False,
            text=caption, font=dict(family=FONT, size=10, color=ink("muted")))
        fig.update_xaxes(visible=False, row=1, col=index)
        fig.update_yaxes(visible=False, row=1, col=index)
    base_layout(fig, 200, "Protocol violations", margin=dict(l=20, r=20, t=54, b=18))
    return fig


def plot_integrity_strip(board):
    integrity = board["integrity"]
    rows = decisions(board["phase"])
    draws = (rows.loc[rows["randomization_draw"].notna(), "randomization_draw"]
             .astype(float).tolist() if not rows.empty else [])
    return {
        "eligibility_rate": plot_integrity_bullet(
            integrity["eligibility_rate"], "Eligibility rate", band=ELIGIBILITY_BAND),
        "send_rate": plot_integrity_bullet(
            integrity["send_rate"], "Send rate",
            target_point=integrity["send_rate"]["target"],
            width_range=(0, max(1.0, (integrity["send_rate"]["value"] or 0) * 1.1))),
        "cap_hit_rate": plot_integrity_bullet(
            integrity["cap_hit_rate"], "Cap-hit rate", band=(0, CAP_HIT_ALARM)),
        "outcome_capture": plot_integrity_bullet(
            integrity["outcome_capture"], "Outcome capture",
            band=(OUTCOME_CAPTURE_ALARM, 1.0)),
        "randomization": plot_randomization_ecdf(integrity, draws),
        "violations": plot_violation_counters(
            integrity, int(daily_grid_metrics(board["phase"]).query("is_run_in")["sent_n"].sum())),
    }

In [327]:
# STAGE 1D — ALERT LOG
SEVERITY_STATUS = {"critical": "critical", "high": "serious", "warning": "warning"}


def style_alert_log(feed=None, include_resolved=False):
    feed = alert_feed(include_resolved) if feed is None else feed
    if feed.empty:
        return pd.DataFrame(columns=["severity", "rule_id", "scope", "fired_at"]).style
    table = feed.copy()
    table["severity"] = [
        f"{STATUS_ICON[SEVERITY_STATUS[s]]} {s}" for s in table["severity"]]
    table["scope"] = [
        f"{'COHORT' if scope == 'cohort' else 'PARTICIPANT'}" for scope in table["scope"]]
    table["participant"] = [
        "-" if pd.isna(uid) else str(int(uid)) for uid in table["user_id"]]
    table["fired_at"] = pd.to_datetime(table["fired_at"], utc=True).dt.tz_convert(
        PARTICIPANT_TZ).dt.strftime("%b %d %H:%M")
    table["detail"] = [json.dumps(payload or {})[:80] for payload in table["payload"]]
    columns = ["severity", "rule_id", "scope", "participant", "fired_at", "link_date", "detail"]

    def tag(value):
        for name, role in SEVERITY_STATUS.items():
            if value.endswith(name):
                return (f"color:{STATUS[role]};font-weight:600;"
                        f"font-family:{FONT};white-space:nowrap")
        return ""

    def scope_tag(value):
        colour = ink("muted") if value == "COHORT" else ink("text_secondary")
        return f"color:{colour};font-size:11px;letter-spacing:.04em"

    return (
        table[columns].style
        .map(tag, subset=["severity"])
        .map(scope_tag, subset=["scope"])
        .set_properties(**{"font-family": FONT, "font-size": "12px",
                           "color": ink("text_secondary"), "background-color": ink("surface")})
        .set_table_styles([
            {"selector": "th", "props": [("font-family", FONT), ("font-size", "11px"),
                                         ("color", ink("muted")), ("text-align", "left"),
                                         ("border-bottom", f"1px solid {ink('axis')}")]},
            {"selector": "td", "props": [("border-bottom", f"1px solid {ink('grid')}")]},
        ])
        .hide(axis="index")
    )

In [328]:
cards = plot_benchmark_cards(board)
cards["slot_coverage"]

In [329]:
cards["prompt_response"]

In [330]:
cards["wear"]

In [331]:
cards["retention"]

In [332]:
cards["hair_sample"]

In [333]:
plot_funnel(board)

In [334]:
strip = plot_integrity_strip(board)
strip["eligibility_rate"]

In [335]:
strip["send_rate"]

In [336]:
strip["cap_hit_rate"]

In [337]:
strip["outcome_capture"]

In [338]:
strip["randomization"]

In [339]:
strip["violations"]

In [340]:
style_alert_log()

severity,rule_id,scope,participant,fired_at,link_date,detail
■ critical,runin_violation,COHORT,-,Sep 16 06:00,2026-09-15,"{""detail"": ""engine has no run-in gate"", ""date"": ""2026-09-15""}"
■ critical,sync_stale,COHORT,-,Sep 16 06:00,2026-09-15,"{""measurable"": false, ""date"": ""2026-09-15""}"
■ critical,cooldown_violation,PARTICIPANT,1007,Sep 16 06:00,2026-09-15,"{""gap_min"": 28, ""date"": ""2026-09-15""}"
■ critical,cap_exceeded,PARTICIPANT,1008,Sep 16 06:00,2026-09-15,"{""delivered_n"": 6, ""date"": ""2026-09-15""}"
▲ high,no_ema_48h,PARTICIPANT,1009,Sep 16 06:00,2026-09-15,"{""hours"": 62, ""date"": ""2026-09-15""}"
▲ high,wear_low,PARTICIPANT,1010,Sep 16 06:00,2026-09-15,"{""consecutive_days"": 3, ""date"": ""2026-09-15""}"
▲ warning,slot_coverage_low,PARTICIPANT,1011,Sep 16 06:00,2026-09-15,"{""coverage"": 0.31, ""date"": ""2026-09-15""}"


In [341]:
benchmark_table(board)

,tile,measurable,suppressed,value,target,wilson_low,wilson_high,numerator,denominator,participants,unit
0,Slot coverage (proxy),True,False,0.488932,0.75,0.463992,0.513927,751.0,1536.0,14,check-in slots covered
1,prompt_response,True,False,0.769231,0.70,0.638662,0.862757,40.0,52.0,10,delivered prompts responded to
2,wear,True,False,0.981928,0.80,0.948220,0.993835,163.0,166.0,14,participant-days meeting the coverage target
3,retention,True,True,NaN,0.85,NaN,NaN,1.0,1.0,1,participants retained at day 35
4,Hair sample,False,False,NaN,0.90,NaN,NaN,NaN,NaN,0,hair samples collected


In [342]:
# STAGE 2/3 PLOTS — PALETTE EXTENSION
# Every value below was run through scripts/validate_palette.js. Three obvious
# choices failed and the design changed because of it; see the notes on each.
PALETTE["light"].update({
    "page": "#f9f9f7",
    # Ordinal floor (step 250), not the full 100-700 sequential range: the pale
    # end of that range sits at 1.29:1 against the surface, which would make a
    # zero cell indistinguishable from a structurally blank one.
    "heatmap": [[0.0, "#86b6ef"], [0.25, "#3987e5"], [0.5, "#256abf"],
                [0.75, "#184f95"], [1.0, "#0d366b"]],
    "series_3": "#1baf7a", "series_4": "#eda100",
    "series_5": "#e87ba4", "series_6": "#008300",
})
PALETTE["dark"].update({
    "page": "#0d0d0d",
    "heatmap": [[0.0, "#b7d3f6"], [0.25, "#6da7ec"], [0.5, "#3987e5"],
                [0.75, "#256abf"], [1.0, "#184f95"]],
    "series_3": "#199e70", "series_4": "#c98500",
    "series_5": "#d55181", "series_6": "#008300",
})

def series_hues(n):
    return [ink(f"series_{index}") for index in range(1, n + 1)]

# Three identities in a part-to-whole bar, not three states. The status trio was
# measured first and fails: good vs critical is dE 4.1 under deuteranopia.
SLOT_CATEGORIES = (
    ("covered", "Covered - check-in submitted", "series_1"),
    ("reminded", "Reminded, not covered - participant skipped", "series_3"),
    ("silent", "Silent - scheduler never fired (not non-compliance)", "series_2"),
)

# One y-row per outcome. As bare scatter marks these five fail the all-pairs
# normal-vision floor (dE 12.9), so position carries identity and hue follows.
DECISION_ROWS = (
    "below within-person threshold",
    "cooldown active",
    "daily cap reached",
    "eligible-not-sent",
    "eligible-sent",
)
DECISION_LABEL = {
    "below within-person threshold": "below threshold",
    "cooldown active": "cooldown active",
    "daily cap reached": "cap reached",
    "eligible-not-sent": "eligible, not sent",
    "eligible-sent": "eligible, sent",
}
COMPLETENESS_STATES = ("answered", "not answered", "not applicable")
METRIC_LABEL = {
    "slot_coverage": "Slot coverage (0-1)",
    "wear": "Wear coverage (0-1)",
    "delivered_n": "Prompts delivered (count)",
    "completeness_mean": "Item completeness (0-1)",
}
CLOCK_TICKS = list(range(0, MINUTES_PER_DAY + 1, 180))
CLOCK_LABELS = [f"{minute // 60:02d}:00" for minute in CLOCK_TICKS]

In [343]:
# STAGE 2A — CROSS-PARTICIPANT HEATMAP
def plot_grid(metric="slot_coverage", phase="all", axis="study_day"):
    payloads = {name: grid(name, phase, axis) for name in GRID_METRICS}
    base = payloads[metric]
    rows = base["rows"]
    labels = [str(row["user_id"]) for row in rows]
    columns = base["columns"]
    x = [str(column) for column in columns]

    fig = go.Figure()
    for name in GRID_METRICS:
        payload = payloads[name]
        zmax = DAILY_PROMPT_CAP if name == "delivered_n" else 1.0
        fig.add_trace(go.Heatmap(
            z=[row["values"] for row in payload["rows"]],
            x=x, y=labels, zmin=0, zmax=zmax,
            colorscale=ink("heatmap"), visible=(name == metric),
            xgap=2, ygap=2, hoverongaps=False,
            colorbar=dict(title=dict(text=METRIC_LABEL[name], side="right", font=dict(size=11)),
                          thickness=10, outlinewidth=0, tickfont=dict(size=10)),
            customdata=[[str(date) for date in row["local_dates"]] for row in payload["rows"]],
            hovertemplate=("participant %{y}<br>" + ("study day" if axis == "study_day" else "date")
                           + " %{x}<br>%{customdata}<br>"
                           + METRIC_LABEL[name] + ": %{z}<extra></extra>")))

    if axis == "study_day":
        run_in_end = min(RUN_IN_DAYS, len(columns)) - 0.5
        fig.add_shape(type="rect", x0=-0.5, x1=run_in_end, y0=-0.5, y1=len(rows) - 0.5,
                      line=dict(width=0), fillcolor=ink("band"), opacity=0.55, layer="below")
        fig.add_annotation(x=(run_in_end - 0.5) / 2, y=len(rows) - 0.5, yanchor="bottom",
                           text="run-in (days 0-6, no prompts expected)", showarrow=False,
                           font=dict(size=10, color=ink("muted")))

    alert_x, alert_y, silent_x, silent_y = [], [], [], []
    for index, row in enumerate(rows):
        for position, column in enumerate(columns):
            if row["values"][position] is None and not row["alert"][position]:
                continue
            if row["alert"][position]:
                alert_x.append(x[position]); alert_y.append(labels[index])
            if row["silent"][position]:
                silent_x.append(x[position]); silent_y.append(labels[index])

    if silent_x:
        fig.add_trace(go.Scatter(
            x=silent_x, y=silent_y, mode="markers", name="silent slot",
            marker=dict(symbol="line-ne", size=11, color=STATUS["critical"],
                        line=dict(width=2, color=STATUS["critical"])),
            hovertemplate="scheduler silent<extra></extra>", showlegend=True))
    if alert_x:
        fig.add_trace(go.Scatter(
            x=alert_x, y=alert_y, mode="markers", name="open alert",
            marker=dict(symbol="square", size=6, color=STATUS["critical"],
                        line=dict(width=1, color=ink("surface"))),
            hovertemplate="open alert<extra></extra>", showlegend=True))

    buttons = []
    for index, name in enumerate(GRID_METRICS):
        visible = [position == index for position in range(len(GRID_METRICS))]
        visible += [True] * (len(fig.data) - len(GRID_METRICS))
        buttons.append(dict(label=METRIC_LABEL[name], method="update",
                            args=[{"visible": visible}]))

    height = max(320, 34 * len(rows) + 170)
    base_layout(fig, height,
                f"Participant grid - {phase} ({'study day' if axis == 'study_day' else 'calendar date'})",
                margin=dict(l=90, r=110, t=96, b=44))
    # Structural blanks fall through to the plot background, so it is the page
    # plane rather than the surface: a day outside the window never reads as a
    # pale ramp step, which is what a zero looks like.
    fig.update_layout(
        plot_bgcolor=ink("page"), showlegend=bool(silent_x or alert_x),
        legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom",
                    font=dict(size=10, color=ink("text_secondary"))),
        updatemenus=[dict(buttons=buttons, direction="down", showactive=True,
                          x=1.0, xanchor="right", y=1.16, yanchor="top",
                          bgcolor=ink("surface"), bordercolor=ink("axis"),
                          font=dict(size=11, color=ink("text")))])
    fig.update_xaxes(title=dict(text="study day" if axis == "study_day" else "local date",
                                font=dict(size=11, color=ink("muted"))),
                     type="category", tickangle=0, nticks=18)
    fig.update_yaxes(type="category", autorange="reversed",
                     title=dict(text="participant (risk desc)",
                                font=dict(size=11, color=ink("muted"))))
    return fig

In [344]:
# STAGE 2B — SLOT BREAKDOWN
def plot_slot_split(phase="all", days=RISK_TRAILING_DAYS):
    split = slot_split(phase, days)
    fig = go.Figure()
    if split.empty:
        fig.add_annotation(x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False,
                           text="no active participant-days in the window",
                           font=dict(size=12, color=ink("muted")))
        base_layout(fig, 200, f"Slot breakdown - trailing {days} days")
        return fig

    split = split.sort_values("silent_pct", ascending=False)
    labels = [str(user_id) for user_id in split["user_id"]]
    for column, legend, role in SLOT_CATEGORIES:
        colour = ink(role)
        fig.add_trace(go.Bar(
            x=split[f"{column}_pct"], y=labels, orientation="h", name=legend,
            marker=dict(color=colour, line=dict(color=ink("surface"), width=2)),
            text=[f"{value:.0%}" if value >= 0.08 else "" for value in split[f"{column}_pct"]],
            textposition="inside", insidetextanchor="middle",
            textfont=dict(size=10, color=ink("surface")),
            customdata=split[column],
            hovertemplate=(f"participant %{{y}}<br>{legend}<br>"
                           "%{customdata} slots (%{x:.0%})<extra></extra>")))

    base_layout(fig, max(260, 30 * len(split) + 150),
                f"Check-in slots by outcome - trailing {days} days",
                margin=dict(l=90, r=40, t=120, b=44))
    fig.update_layout(barmode="stack", bargap=0.3, showlegend=True,
                      legend=dict(orientation="h", y=1.04, x=0, yanchor="bottom",
                                  font=dict(size=10, color=ink("text_secondary"))))
    fig.update_xaxes(range=[0, 1], tickformat=".0%",
                     title=dict(text="share of the day's six slots",
                                font=dict(size=11, color=ink("muted"))))
    fig.update_yaxes(type="category",
                     title=dict(text="participant", font=dict(size=11, color=ink("muted"))))
    return fig

In [345]:
# STAGE 2C/2D — RISK CONTRIBUTIONS AND THE RAIL
RISK_TERM_LABEL = {
    "ema_stale": "days since last check-in",
    "low_coverage": "low slot coverage",
    "sync_stale": "stale sync",
    "low_wear": "low wear",
    "missing_signal": "missing B1/B2",
    "open_critical": "open alert",
}


def plot_risk_contributions(phase="all"):
    scores = risk_scores(phase)
    scored = scores[scores["risk_score"].notna()]
    fig = go.Figure()
    if scored.empty:
        fig.add_annotation(x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False,
                           text="no participant is currently in run-in or MRT",
                           font=dict(size=12, color=ink("muted")))
        base_layout(fig, 200, "Risk score contributions")
        return fig

    scored = scored.sort_values("risk_score")
    labels = [str(user_id) for user_id in scored["user_id"]]
    terms = list(RISK_WEIGHTS)
    for term, colour in zip(terms, series_hues(len(terms))):
        values = [components.get(term, 0) for components in scored["risk_components"]]
        fig.add_trace(go.Bar(
            x=values, y=labels, orientation="h", name=RISK_TERM_LABEL[term],
            marker=dict(color=colour, line=dict(color=ink("surface"), width=2)),
            # Three of these hues sit below 3:1 on the light surface, so the
            # relief rule applies: every segment carries its own value.
            text=[str(value) if value else "" for value in values],
            textposition="inside", insidetextanchor="middle",
            textfont=dict(size=10, color=ink("surface")),
            hovertemplate=(f"participant %{{y}}<br>{RISK_TERM_LABEL[term]}"
                           "<br>%{x} points<extra></extra>")))

    # A trace, not annotations: annotations anchored to a category axis do not
    # participate in its category list and drag the layout apart.
    fig.add_trace(go.Scatter(
        x=list(scored["risk_score"]), y=labels, mode="text",
        text=[f"<b>{int(total)}</b>" for total in scored["risk_score"]],
        textposition="middle right", textfont=dict(size=11, color=ink("text")),
        hoverinfo="skip", showlegend=False))

    base_layout(fig, max(260, 32 * len(scored) + 150), f"Risk score contributions - {phase}",
                margin=dict(l=90, r=70, t=120, b=44))
    fig.update_layout(barmode="stack", bargap=0.3, showlegend=True,
                      legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom",
                                  font=dict(size=10, color=ink("text_secondary"))))
    fig.update_xaxes(title=dict(text="weighted points", font=dict(size=11, color=ink("muted"))),
                     rangemode="tozero")
    fig.update_yaxes(type="category",
                     title=dict(text="participant", font=dict(size=11, color=ink("muted"))))
    return fig


def style_participant_rail(user_id, phase="all"):
    rail = participant_rail(user_id, phase)
    sync = ("no writer - unmeasurable" if not rail["sync_measurable"]
            else "-" if rail["last_sync_age_h"] is None
            else f"{rail['last_sync_age_h']:.1f} h ago")
    coverage = (f"{rail['slot_coverage_num']}/{rail['slot_coverage_den']}"
                + (f" ({rail['slot_coverage_rate']:.0%})" if rail["slot_coverage_rate"] is not None
                   else "  rate withheld"))
    rows = [
        ("participant", str(rail["user_id"])),
        ("phase", rail["phase"] or "-"),
        ("study day", f"{rail['study_day']} of {STUDY_DAYS - 1}"),
        ("days remaining", str(rail["days_remaining"])),
        ("risk score", "-" if rail["risk_score"] is None else str(int(rail["risk_score"]))),
        ("last check-in", "-" if rail["last_ema_at"] is None
         else pd.Timestamp(rail["last_ema_at"]).strftime("%b %d %H:%M")),
        ("last sync", sync),
        ("slot coverage", coverage),
        ("wear days met", f"{rail['wear_days_met']}/{rail['wear_days_scored']}"),
        ("prompts delivered", str(rail["prompts_delivered"])),
        ("open alerts", str(len(rail["open_alerts"]))),
    ]
    table = pd.DataFrame(rows, columns=["field", "value"])
    return (table.style
            .set_properties(**{"font-family": FONT, "font-size": "12px",
                               "color": ink("text"), "background-color": ink("surface")})
            .set_properties(subset=["field"], **{"color": ink("muted"), "font-size": "11px"})
            .set_table_styles([
                {"selector": "th", "props": [("display", "none")]},
                {"selector": "td", "props": [("border-bottom", f"1px solid {ink('grid')}"),
                                             ("padding", "4px 10px")]}])
            .hide(axis="index"))

In [346]:
# STAGE 3A — SIX-LANE DAILY STRIP
# Lane 1 is minute-resolution: wear_lane bins to the minute, which is what
# wear_coverage scores against, and the fixture stores one sample per minute.
LANE_TITLES = ("Wear", "Sync", "Check-ins", "Decision points", "Delivery", "MSSD")


def _ref(kind, row):
    """Plotly names the first subplot axis 'x'/'y', not 'x1'/'y1'."""
    return kind if row == 1 else f"{kind}{row}"


def _clock_axis(fig, row, last):
    fig.update_xaxes(range=[0, MINUTES_PER_DAY], tickvals=CLOCK_TICKS,
                     ticktext=CLOCK_LABELS if last else [""] * len(CLOCK_TICKS),
                     showgrid=True, gridcolor=ink("grid"), row=row, col=1)


def _lane_wear(fig, lane, row):
    start, end = lane["waking_window"]
    fig.add_shape(type="rect", x0=start, x1=end, y0=0, y1=1,
                  xref=_ref("x", row), yref=_ref("y", row) + " domain",
                  fillcolor=ink("band"), opacity=1.0, line=dict(width=0),
                  layer="below")
    for edge in (start, end):
        fig.add_shape(type="line", x0=edge, x1=edge, y0=0, y1=1,
                      xref=_ref("x", row), yref=_ref("y", row) + " domain",
                      line=dict(color=ink("axis"), width=1, dash="dot"))
    fig.add_annotation(x=start, y=1.0, xref=_ref("x", row), yref=_ref("y", row) + " domain",
                       xanchor="left", yanchor="bottom", text="waking window 08:00-22:00",
                       showarrow=False, font=dict(size=9, color=ink("muted")))
    if not lane["has_data"]:
        fig.add_annotation(x=MINUTES_PER_DAY / 2, y=0.5, xref=_ref("x", row), yref=_ref("y", row) + " domain",
                           text="no heart-rate samples", showarrow=False,
                           font=dict(size=10, color=ink("muted")))
        return
    bins = lane["bins"]
    fig.add_trace(go.Heatmap(
        z=[[1 if value else 0 for value in bins]],
        x=list(range(MINUTES_PER_DAY)), y=["worn"],
        colorscale=[[0.0, "rgba(0,0,0,0)"], [1.0, ink("series_1")]],
        showscale=False, hovertemplate="minute %{x}<extra></extra>"), row=row, col=1)
    for gap in lane["gaps_gt_2h"]:
        clipped_start, clipped_end = max(gap["start"], start), min(gap["end"], end)
        if clipped_end - clipped_start <= 0:
            continue
        fig.add_shape(type="rect", x0=clipped_start, x1=clipped_end, y0=0, y1=1,
                      xref=_ref("x", row), yref=_ref("y", row) + " domain",
                      fillcolor=STATUS["critical"], opacity=0.28, line=dict(width=0))
        fig.add_annotation(x=(clipped_start + clipped_end) / 2, y=0.82,
                           xref=_ref("x", row), yref=_ref("y", row) + " domain",
                           text=f"{clipped_end - clipped_start} min", showarrow=False,
                           font=dict(size=9, color=STATUS["critical"]))


def _lane_sync(fig, lane, local_date, row):
    if not lane["advances"] and lane["carried_in"] is None:
        message = ("no sync writer - unmeasurable" if not lane["measurable"]
                   else "no sync advance recorded")
        fig.add_annotation(x=MINUTES_PER_DAY / 2, y=0.5, xref=_ref("x", row), yref=_ref("y", row) + " domain",
                           text=message, showarrow=False,
                           font=dict(size=10, color=ink("muted")))
        return
    minutes, lags = [], []
    for advance in lane["advances"]:
        observed = advance["minute"]
        synced = local_minutes(advance["last_synced_at"], local_date)
        if observed is None:
            continue
        minutes.append(observed)
        lags.append(None if synced is None else max(0.0, observed - synced))
    fig.add_trace(go.Scatter(
        x=minutes, y=lags, mode="lines+markers", line=dict(color=ink("series_1"), width=2,
                                                           shape="hv"),
        marker=dict(size=8, color=ink("series_1"), line=dict(color=ink("surface"), width=2)),
        hovertemplate="sync at %{x:.0f} min<br>clock %{y:.0f} min behind<extra></extra>",
        showlegend=False), row=row, col=1)


def _lane_checkins(fig, lane, row):
    if not lane["emas"] and not lane["reminders"]:
        fig.add_annotation(x=MINUTES_PER_DAY / 2, y=0.5, xref=_ref("x", row),
                           yref=_ref("y", row) + " domain",
                           text="no check-in submitted and no reminder sent", showarrow=False,
                           font=dict(size=10, color=ink("muted")))
    for slot in lane["slots"]:
        fig.add_shape(type="line", x0=slot["start"], x1=slot["start"], y0=0, y1=1,
                      yref=_ref("y", row) + " domain", line=dict(color=ink("grid"), width=1, dash="dash"))
    if lane["emas"]:
        fig.add_trace(go.Scatter(
            x=[mark["minute"] for mark in lane["emas"]],
            y=[mark["ema_type"] for mark in lane["emas"]], mode="markers",
            marker=dict(size=12, line=dict(color=ink("surface"), width=2),
                        color=[0 if mark["completeness"] is None else mark["completeness"]
                               for mark in lane["emas"]],
                        colorscale=ink("heatmap"), cmin=0, cmax=1, showscale=False),
            customdata=[[mark["ema_id"],
                         "-" if mark["completeness"] is None else f"{mark['completeness']:.0%}",
                         "yes" if mark["missing_b1b2"] else "no"] for mark in lane["emas"]],
            hovertemplate=("EMA %{customdata[0]} (%{y})<br>completeness %{customdata[1]}"
                           "<br>missing B1/B2: %{customdata[2]}<extra></extra>"),
            showlegend=False), row=row, col=1)
    if lane["reminders"]:
        fig.add_trace(go.Scatter(
            x=[tick["minute"] for tick in lane["reminders"]],
            y=["reminder"] * len(lane["reminders"]), mode="markers+text",
            marker=dict(symbol="line-ns", size=10, line=dict(color=ink("series_2"), width=2)),
            text=[str(tick["daily_count_at_send"]) for tick in lane["reminders"]],
            textposition="middle right", textfont=dict(size=9, color=ink("muted")),
            hovertemplate="reminder, slot index %{text}<extra></extra>",
            showlegend=False), row=row, col=1)


def _lane_decisions(fig, points, row):
    hues = dict(zip(DECISION_ROWS, series_hues(len(DECISION_ROWS))))
    for outcome in DECISION_ROWS:
        marks = [point for point in points if point["outcome"] == outcome]
        fig.add_trace(go.Scatter(
            x=[mark["minute"] for mark in marks],
            y=[DECISION_LABEL[outcome]] * len(marks), mode="markers",
            marker=dict(size=11, color=hues[outcome],
                        line=dict(color=ink("surface"), width=2)),
            customdata=[[mark["decision_point_id"],
                         "-" if mark["observed_mssd"] is None else round(mark["observed_mssd"], 3),
                         "-" if mark["randomization_draw"] is None
                         else round(mark["randomization_draw"], 3)] for mark in marks],
            hovertemplate=("%{customdata[0]}<br>MSSD %{customdata[1]}"
                           "<br>draw %{customdata[2]}<extra></extra>"),
            showlegend=False), row=row, col=1)
    fig.add_trace(go.Scatter(
        x=[None] * len(DECISION_ROWS),
        y=[DECISION_LABEL[outcome] for outcome in DECISION_ROWS],
        mode="markers", marker=dict(size=0.1, color="rgba(0,0,0,0)"),
        hoverinfo="skip", showlegend=False), row=row, col=1)
    other = [point for point in points if point["outcome"] not in DECISION_ROWS]
    if other:
        fig.add_trace(go.Scatter(
            x=[mark["minute"] for mark in other], y=["other"] * len(other), mode="markers",
            marker=dict(size=11, color=ink("muted"), line=dict(color=ink("surface"), width=2)),
            hovertemplate="%{text}<extra></extra>",
            text=[mark["outcome"] for mark in other], showlegend=False), row=row, col=1)


def _lane_delivery(fig, prompts, row):
    if not prompts:
        fig.add_annotation(x=MINUTES_PER_DAY / 2, y=0.5, xref=_ref("x", row), yref=_ref("y", row) + " domain",
                           text="no prompt sent", showarrow=False,
                           font=dict(size=10, color=ink("muted")))
        return
    for index, prompt in enumerate(prompts):
        label = f"#{prompt['jitai_log_id']}"
        window = prompt["outcome_window"]
        if window[0] is not None and window[1] is not None:
            fig.add_shape(type="rect", x0=window[0], x1=window[1], y0=index - 0.35,
                          y1=index + 0.35, fillcolor=ink("series_1"), opacity=0.12,
                          line=dict(width=0), layer="below")
        span = [value for value in (prompt["push_sent_minute"],
                                    prompt["device_received_minute"],
                                    prompt["receipt_reported_minute"]) if value is not None]
        if span:
            fig.add_trace(go.Scatter(
                x=[min(span), max(span) + 1], y=[label, label], mode="lines",
                line=dict(color=ink("series_1"), width=8),
                hovertemplate=(f"{label}<br>{prompt['delivery_status']}"
                               f"<br>{prompt['receipt_platform'] or 'unknown'} /"
                               f" {prompt['receipt_app_state'] or '-'}<extra></extra>"),
                showlegend=False), row=row, col=1)
        if prompt["delivery_status"] == "failed":
            fig.add_trace(go.Scatter(
                x=[prompt["push_sent_minute"]], y=[label], mode="markers",
                marker=dict(symbol="x", size=11, color=STATUS["critical"]),
                hovertemplate=f"{prompt['delivery_error']}<extra></extra>",
                showlegend=False), row=row, col=1)
        if prompt["linked_ema_responded_minute"] is not None:
            fig.add_trace(go.Scatter(
                x=[prompt["linked_ema_responded_minute"]], y=[label], mode="markers",
                marker=dict(symbol="diamond", size=10, color=ink("series_3"),
                            line=dict(color=ink("surface"), width=2)),
                hovertemplate="post-prompt check-in<extra></extra>",
                showlegend=False), row=row, col=1)


def _lane_mssd(fig, points, row):
    if not points:
        fig.add_annotation(x=MINUTES_PER_DAY / 2, y=0.5, xref=_ref("x", row), yref=_ref("y", row) + " domain",
                           text="no decision point", showarrow=False,
                           font=dict(size=10, color=ink("muted")))
        return
    ordered = sorted(points, key=lambda point: point["minute"] or 0)
    fig.add_trace(go.Scatter(
        x=[point["minute"] for point in ordered],
        y=[point["observed_mssd"] for point in ordered], mode="lines+markers",
        line=dict(color=ink("series_1"), width=2, shape="hv"),
        marker=dict(size=8, color=ink("series_1"), line=dict(color=ink("surface"), width=2)),
        name="observed MSSD",
        hovertemplate="observed %{y:.3f}<extra></extra>", showlegend=False), row=row, col=1)
    fig.add_trace(go.Scatter(
        x=[point["minute"] for point in ordered],
        y=[point["threshold"] for point in ordered], mode="lines",
        line=dict(color=ink("muted"), width=2, dash="dot", shape="hv"),
        name="threshold", hovertemplate="threshold %{y:.3f}<extra></extra>",
        showlegend=False), row=row, col=1)
    flagged = [point for point in ordered if point["unexplained"]]
    if flagged:
        fig.add_trace(go.Scatter(
            x=[point["minute"] for point in flagged],
            y=[point["observed_mssd"] for point in flagged], mode="markers",
            marker=dict(symbol="star", size=14, color=STATUS["critical"],
                        line=dict(color=ink("surface"), width=2)),
            hovertemplate="over threshold but no draw taken<extra></extra>",
            showlegend=False), row=row, col=1)


def plot_timeline_day(user_id, local_date):
    day = {
        "wear": wear_lane(user_id, local_date),
        "sync": sync_lane(user_id, local_date),
        "checkins": checkin_lane(user_id, local_date),
        "decisions": decision_lane(user_id, local_date),
        "delivery": delivery_lane(user_id, local_date),
        "mssd": mssd_lane(user_id, local_date),
    }
    fig = make_subplots(rows=6, cols=1, shared_xaxes=True, vertical_spacing=0.035,
                        row_heights=[0.10, 0.13, 0.19, 0.20, 0.20, 0.18],
                        subplot_titles=LANE_TITLES)
    _lane_wear(fig, day["wear"], 1)
    _lane_sync(fig, day["sync"], local_date, 2)
    _lane_checkins(fig, day["checkins"], 3)
    _lane_decisions(fig, day["decisions"], 4)
    _lane_delivery(fig, day["delivery"], 5)
    _lane_mssd(fig, day["mssd"], 6)

    for row in range(1, 7):
        _clock_axis(fig, row, last=(row == 6))
        fig.update_yaxes(showgrid=False, tickfont=dict(size=10, color=ink("muted")),
                         row=row, col=1)
    fig.update_yaxes(showticklabels=False, row=1, col=1)
    for row, key in ((3, "checkins"), (4, "decisions"), (5, "delivery"), (6, "mssd")):
        empty = not day[key] if key != "checkins" else not (
            day["checkins"]["emas"] or day["checkins"]["reminders"])
        if empty:
            fig.update_yaxes(showticklabels=False, row=row, col=1)
    fig.update_yaxes(title=dict(text="min behind", font=dict(size=9, color=ink("muted"))),
                     rangemode="tozero", row=2, col=1)
    fig.update_yaxes(categoryorder="array",
                     categoryarray=[DECISION_LABEL[name] for name in DECISION_ROWS] + ["other"],
                     row=4, col=1)

    study_day = study_day_for_local(user_id, local_date)
    base_layout(fig, 900,
                f"Participant {user_id} - {local_date} (study day {study_day})",
                margin=dict(l=120, r=40, t=80, b=44))
    fig.update_layout(plot_bgcolor=ink("surface"), hovermode="closest")
    for annotation in fig.layout.annotations[:len(LANE_TITLES)]:
        annotation.update(x=0, xanchor="left", font=dict(size=11, color=ink("text")))
    return fig


def study_day_for_local(user_id, local_date):
    anchor = day1_dates().get(user_id)
    return None if anchor is None else (local_date - anchor).days


def plot_timeline(user_id, days=7, end=None):
    end = end or today_local()
    return [plot_timeline_day(user_id, end - timedelta(days=offset))
            for offset in reversed(range(days))]

In [347]:
# STAGE 3B — DELIVERY FUNNEL
def plot_delivery_funnel(user_id):
    payload = delivery_funnel(user_id)
    stages = payload["stages"]
    fig = go.Figure(go.Funnel(
        orientation="h", y=list(stages["stage"]), x=list(stages["n"]),
        marker=dict(color=ink("series_1"), line=dict(color=ink("surface"), width=2)),
        connector=dict(line=dict(color=ink("axis"), width=1, dash="dot")),
        text=[f"<b>{int(n)}</b>" + ("" if pd.isna(pct) or not pct else f"  -{pct:.0%}")
              for n, pct in zip(stages["n"], stages["drop_pct"])],
        textposition="inside", textfont=dict(size=12, color=ink("surface")),
        textinfo="text",
        hovertemplate="%{y}<br>%{x} prompts<extra></extra>"))
    for index, row in enumerate(stages.to_dict("records")):
        if not row["drop_from_previous"] or pd.isna(row["drop_from_previous"]):
            continue
        fig.add_annotation(
            x=1.0, xref="paper", xanchor="left", y=row["stage"], xshift=6, showarrow=False,
            text=(f"<span style='color:{STATUS['critical']}'>"
                  f"-{int(row['drop_from_previous'])}</span>"),
            font=dict(size=11))
    base_layout(fig, 340, f"Delivery funnel - participant {user_id}",
                margin=dict(l=170, r=80, t=80, b=40))
    return fig


def plot_delivery_splits(user_id):
    """Platform and app state as two charts. They measure different things, so
    one chart with two scales would invent a relationship that is not there."""
    payload = delivery_funnel(user_id)
    figures = {}
    for column, title in (("receipt_platform", "Delivered by platform"),
                          ("receipt_app_state", "Delivered by app state")):
        counts = payload["splits"].get(column) or {}
        fig = go.Figure()
        if counts:
            keys = list(counts)
            fig.add_trace(go.Bar(
                x=keys, y=[counts[key] for key in keys],
                marker=dict(color=series_hues(max(len(keys), 1))[:len(keys)],
                            line=dict(color=ink("surface"), width=2)),
                text=[counts[key] for key in keys], textposition="outside",
                textfont=dict(size=11, color=ink("text")),
                hovertemplate="%{x}<br>%{y} delivered<extra></extra>", showlegend=False))
        else:
            fig.add_annotation(x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False,
                               text="nothing delivered", font=dict(size=11, color=ink("muted")))
        base_layout(fig, 250, title, margin=dict(l=60, r=40, t=70, b=50))
        fig.update_yaxes(rangemode="tozero", showgrid=True, gridcolor=ink("grid"))
        figures[column] = fig
    return figures

In [348]:
# STAGE 3C — ITEM COMPLETENESS MATRIX
def plot_completeness_matrix(user_id, local_date):
    matrix = completeness_matrix(user_id, local_date)
    fig = go.Figure()
    if matrix.empty:
        fig.add_annotation(x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False,
                           text="no check-in submitted that day",
                           font=dict(size=12, color=ink("muted")))
        base_layout(fig, 200, f"Item completeness - participant {user_id}, {local_date}")
        return fig

    meta = ["ema_id", "ema_type", "completeness", "missing_b1b2", "item_bank_version"]
    columns = [column for column in matrix.columns if column not in meta]
    state_index = {state: index for index, state in enumerate(COMPLETENESS_STATES)}
    z = [[state_index.get(row[column], 2) for column in columns]
         for row in matrix.to_dict("records")]
    labels = [f"{row['ema_id']} ({row['ema_type'].replace('_', ' ')})"
              for row in matrix.to_dict("records")]

    # answered / not answered are a validated all-pairs pair; not applicable is a
    # neutral, never a third hue - it means the branch closed, not a failure.
    scale = [[0.0, ink("series_1")], [0.33, ink("series_1")],
             [0.34, STATUS["critical"]], [0.66, STATUS["critical"]],
             [0.67, ink("grid")], [1.0, ink("grid")]]
    fig.add_trace(go.Heatmap(
        z=z, x=columns, y=labels, zmin=0, zmax=2, colorscale=scale, showscale=False,
        xgap=2, ygap=2,
        text=[[row[column] for column in columns] for row in matrix.to_dict("records")],
        hovertemplate="%{y}<br>%{x}<br>%{text}<extra></extra>"))

    for column in SIGNAL_SUB_ITEMS:
        if column not in columns:
            continue
        position = columns.index(column)
        fig.add_shape(type="rect", x0=position - 0.5, x1=position + 0.5,
                      y0=-0.5, y1=len(labels) - 0.5,
                      line=dict(color=STATUS["critical"], width=2), fillcolor="rgba(0,0,0,0)")

    for index, state in enumerate(COMPLETENESS_STATES):
        colour = [ink("series_1"), STATUS["critical"], ink("grid")][index]
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers", name=state,
            marker=dict(size=10, color=colour, line=dict(color=ink("axis"), width=1)),
            showlegend=True))

    missing = int(matrix["missing_b1b2"].fillna(False).sum())
    base_layout(fig, max(320, 46 * len(labels) + 220),
                f"Item completeness - participant {user_id}, {local_date}",
                margin=dict(l=210, r=40, t=104, b=150))
    fig.add_annotation(
        x=1.0, y=1.0, xref="paper", yref="paper", xanchor="right", yanchor="bottom",
        yshift=26, showarrow=False,
        text=(f"<span style='color:{STATUS['critical']}'>{STATUS_ICON['critical']}</span> "
              f"{missing} check-in(s) missing B1/B2 - no decision point produced"
              if missing else
              f"<span style='color:{STATUS['good']}'>{STATUS_ICON['good']}</span> "
              "B1/B2 present on every check-in"),
        font=dict(family=FONT, size=11, color=ink("text")))
    fig.update_layout(plot_bgcolor=ink("page"), showlegend=True,
                      legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom",
                                  font=dict(size=10, color=ink("text_secondary"))))
    fig.update_xaxes(tickangle=-60, tickfont=dict(size=9, color=ink("muted")))
    fig.update_yaxes(autorange="reversed", tickfont=dict(size=10, color=ink("text")))
    return fig

In [349]:
plot_grid("slot_coverage", "all")

In [350]:
plot_grid("wear", "all", axis="local_date")

In [351]:
plot_slot_split("all")

In [352]:
plot_risk_contributions("all")

In [353]:
style_participant_rail(GRID_USER)

field,value
participant,1003
phase,mrt
study day,18 of 34
days remaining,16
risk score,31
last check-in,-
last sync,29.8 h ago
slot coverage,0/114 rate withheld
wear days met,14/14
prompts delivered,0


In [354]:
# The riskiest participant is often the one with nothing in their lanes; pick the
# busiest instead, and the day that actually has decision points to look at.
TIMELINE_USER = int(jitai_log_df["user_id"].value_counts().idxmax())
TIMELINE_DATE = max(
    (day["local_date"] for day in timeline(TIMELINE_USER, days=14)
     if day["decisions"] and day["checkins"]["emas"]),
    default=today_local())
plot_timeline_day(TIMELINE_USER, TIMELINE_DATE)

In [355]:
plot_delivery_funnel(TIMELINE_USER)

In [356]:
splits = plot_delivery_splits(TIMELINE_USER)
splits["receipt_platform"]

In [357]:
splits["receipt_app_state"]

In [358]:
plot_completeness_matrix(TIMELINE_USER, TIMELINE_DATE)